# ARC3 019 — FP8 PLE feasibility

GPU serving microbench; BF16 → FP8 PLE → BF16; hard limit 80 minutes, not a target runtime.

Research account: **jiyujia**. Private, offline. No competition submission,
game engine, benchmark pickle, or execution of model-generated code.
Preserves the [Franzen Milestone 2 v3](https://www.kaggle.com/code/dfranzen/arc-agi-3-milestone-2-solution)
Pennyroyal launcher and independent drafter. Source license attribution remains
Apache-2.0 for the harness; model and third-party licenses remain their own.

## Method and boundaries
CPU checks compare mounted metadata, sampled PLE dequantization and non-PLE bytes.
The GPU mode uses the same frozen requests, seed, context, cache and service settings.
Synthetic exact-answer quality and tool format are smoke checks, not hidden-game quality.
Loading order/page cache/JIT effects are recorded. High GPU utilization alone is no speedup.
Outputs are written under `019-results`; inputs and responses are retained with hashes.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
if os.environ.get('KAGGLE_IS_COMPETITION_RERUN', '').lower() in {'1', 'true'}:
    raise RuntimeError('Research notebook cannot run as a competition submission')
ROOT_019 = Path('/kaggle/working/019-code')
if ROOT_019.exists():
    raise FileExistsError('Use a fresh Kaggle session for 019')
ROOT_019.mkdir(parents=True)
(ROOT_019 / 'src/arc3_research').mkdir(parents=True)


In [ ]:
%%writefile /kaggle/working/019-code/config.json
{
  "id": "019-fp8-ple",
  "account": "jiyujia",
  "mode": "microbench",
  "phase_order": [
    "bf16_a",
    "fp8",
    "bf16_b"
  ],
  "server_seed": 190018,
  "port": 8001,
  "hard_limit_seconds": 4800,
  "startup_limit_seconds": 900,
  "phase_budget_seconds": 360,
  "weight_audit_limit_seconds": 240,
  "inputs": {
    "bf16": "models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1",
    "fp8": "models/dfranzen/albucino-qwen3.8-flash-next-w4a16-fp8ple/transformers/default/1",
    "draft": "models/dfranzen/albucino-qwen3-8-flash-next-drafter/transformers/default/1",
    "wheelhouse": "datasets/dfranzen/pennyroyal-v253"
  },
  "runtime_source_sha256": {
    "sglang/srt/models/qwen4_exp.py": "35a1785ce4c287821f00f1518b3880ab6838ef58d42fd51ece86d8098b2ed666",
    "sglang/kernels/ops/qwen4_ple.py": "7f64e896c1f00f46ab6c4cd8c5c90cedc528555d1cc7401e9c7a5a3a31b87f16",
    "sglang/srt/environ.py": "84755bded97394a18537d8e329c3d05c15f7ec4aa8a801bd9971f34b4f11d7ed",
    "sglang/srt/model_loader/loader.py": "1016b2c6c2d7eb50ef25e0a731e20096e4b79239bccb6fea26aa59d3f29bf8fe"
  },
  "scope": "Frozen requests, not a game or leaderboard score. PLE precision changes model numerics. Second BF16 controls drift; startup includes cache/JIT order effects, not a clean cold-load estimate."
}


In [ ]:
%%writefile /kaggle/working/019-code/weight-contract.json
{
  "schema_version": 1,
  "experiment": "019",
  "status": "frozen_cpu_contract_not_gpu_validation",
  "created_at_utc": "2026-10-18T10:30:22Z",
  "audit_limits": {
    "max_seconds": 180,
    "parent_process_timeout_seconds": 240,
    "max_bytes_per_source": 134217728,
    "max_metadata_file_bytes": 33554432,
    "max_header_bytes": 33554432,
    "network_allowed": false,
    "complete_weight_tensor_reads_allowed": false
  },
  "expected_text_config": {
    "ple_embed_dim": 2560,
    "heads_per_ngram": 8,
    "ngram_size": 3,
    "ngram_vocab_size_base": 20000000,
    "split_ngram_parts": 128,
    "make_ngram_vocab_size_divisible_by": 128,
    "ple_layer_ids": [
      2
    ],
    "num_hidden_layers": 48
  },
  "identical_target_metadata": [
    "tokenizer.json",
    "tokenizer_config.json",
    "chat_template.jinja",
    "generation_config.json"
  ],
  "expected_tokenizer_sha256": "06b9509352d2af50381ab2247e083b80d32d5c0aba91c272ca9ff729b6a0e523",
  "ple": {
    "prefix": "model.language_model.layers.1.ple.ple_embedding.ngram_embedding.",
    "shard_count": 128,
    "shard_shape": [
      2500012,
      160
    ],
    "total_rows": 320001536,
    "total_elements": 51200245760,
    "bf16_dtype": "BF16",
    "fp8_dtype": "F8_E4M3",
    "pytorch_fp8_dtype": "float8_e4m3fn",
    "scale_key": "model.language_model.layers.1.ple.ple_embedding.ngram_embedding.weight_scale",
    "scale_shape": [
      1
    ],
    "scale_dtype": "BF16",
    "scale_raw_hex": "5139",
    "scale_value": 0.00019931793212890625,
    "scale_semantics": "One global positive direct multiplicative scale: BF16(FP8_value * BF16_scale). No inverse, per-row or per-block scale.",
    "bf16_payload_bytes": 102400491520,
    "fp8_payload_bytes_excluding_scale": 51200245760,
    "resident_location": "Pinned host table via mmap/cudaHostRegister; scale on model device."
  },
  "sampling": {
    "seed": "arc3-019-ple-weight-audit-v1",
    "fixed_rows": [
      "0",
      "floor(rows/2)",
      "rows-1"
    ],
    "random_rows_per_shard": 5,
    "row_selection": "SHA256(seed|full_tensor_key|counter), first 8 bytes little-endian modulo rows; unique rows; stop at 8.",
    "nonple_max_tensors": 96,
    "draft_max_tensors": 16,
    "tensor_span_bytes": 256,
    "tensor_ranges": [
      "first",
      "center",
      "last"
    ],
    "relative_error_bound": 0.07,
    "absolute_error_scale_factor": 0.001953125,
    "max_nrmse": 0.06,
    "allowed_nonfinite_elements": 0,
    "allowed_error_bound_violations": 0,
    "engineering_gate_not_quality_proof": true,
    "bound_rationale": "Conservative engineering smoke gate: 7%*abs(BF16_reference)+scale/512 allows E4M3 quantization and BF16 output rounding. Not a claim of original training-source identity or downstream quality. Stop and inspect violations rather than relax thresholds after seeing results.",
    "ple_rows_per_arm": 1024,
    "ple_elements_per_arm": 163840,
    "ple_payload_bytes_both_arms": 491520,
    "nonple_payload_max_bytes_both_arms": 147456,
    "draft_payload_max_bytes": 12288,
    "total_payload_max_bytes_excluding_scale": 651264
  },
  "stop_conditions": [
    "Missing/ambiguous sources; config/tokenizer/template/key/header mismatch; malformed or overlapping safetensors.",
    "FP8 dtype config absent; missing or wrong 128 shards, shape or dtype; missing, nonpositive, nonfinite or changed global scale.",
    "Sampled non-PLE bytes differ.",
    "Sampled PLE nonfinite, NRMSE above 0.06, or any 7%+scale/512 bound violation.",
    "Read/time budget exceeded or audit incomplete.",
    "Serving must stop if loader downcasts unexpectedly, retains default scale 1, or frozen runtime/config changes.",
    "Do not start serving unless CPU report passed=true. This is not inference-quality validation."
  ],
  "expected_nonple_tensor_count": 222587,
  "expected_nonple_keys_sha256": "63ca678db19cce48475aeb61c005bf804f1dc407bd94d5d4f6a8c849f2aecd62",
  "inputs": {
    "bf16": {
      "kaggle_source": "dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/Transformers/default/1",
      "canonical_path": "/kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1",
      "native_model_version_id": 1007514,
      "hf_repo": "Intel/Qwen3.8-Flash-Next-W4A16-AutoRound",
      "hf_revision": "4c67bf686b7f7fd386bae6b07ab59e8ff1d5b897",
      "kaggle_mirror_payload_verified": false,
      "physical_index_warning": "Kaggle has 22 re-sharded PLE files; HF index points to model-00016-of-00017. Read each mounted index; never transplant the HF file map."
    },
    "fp8": {
      "kaggle_source": "dfranzen/albucino-qwen3.8-flash-next-w4a16-fp8ple/Transformers/default/1",
      "canonical_path": "/kaggle/input/models/dfranzen/albucino-qwen3.8-flash-next-w4a16-fp8ple/transformers/default/1",
      "native_model_version_id": 1026228,
      "hf_repo": "albucino/Qwen3.8-Flash-Next-W4A16-FP8PLE",
      "hf_revision": "c6af9fee7ad2cb8ef44b534a99b8c1b81fb0f6da",
      "kaggle_mirror_payload_verified": false
    },
    "draft": {
      "kaggle_source": "dfranzen/albucino-qwen3-8-flash-next-drafter/Transformers/default/1",
      "canonical_path": "/kaggle/input/models/dfranzen/albucino-qwen3-8-flash-next-drafter/transformers/default/1",
      "quantization": "compressed-tensors symmetric INT4/group32",
      "same_source_for_all_arms": true
    }
  },
  "upstream_header_observation": {
    "observed_at_utc": "2026-10-18T10:30:22Z",
    "method": "Strict HTTP206 and exact Content-Range; 8-byte length then bounded JSON header, plus only two scale payload bytes. No complete weights downloaded.",
    "fp8_headers": [
      {
        "file": "model-plefp8-00000.safetensors",
        "header_size": 2231,
        "header_sha256": "f7a9c23ef7801202a21a27504b5c7b1c2e41fdfd6b9de8b3ad5649e675ff4587",
        "weight_tensor_count": 13
      },
      {
        "file": "model-plefp8-00001.safetensors",
        "header_size": 2234,
        "header_sha256": "2ee6f589ff1f106293439aaa8505abb5c908400443b50b9b2f4ae02523050d5d",
        "weight_tensor_count": 13
      },
      {
        "file": "model-plefp8-00002.safetensors",
        "header_size": 2230,
        "header_sha256": "cb8625bb7050650bcf679a5bd8a8d3bdd1444d5dc3d32f318fb8f80a63754dac",
        "weight_tensor_count": 13
      },
      {
        "file": "model-plefp8-00003.safetensors",
        "header_size": 2221,
        "header_sha256": "840e83616e82ad05f4b82451f2c67c21e030ff02362f298b82711301f2268787",
        "weight_tensor_count": 13
      },
      {
        "file": "model-plefp8-00004.safetensors",
        "header_size": 2222,
        "header_sha256": "9fd53ae738f9d247e55e332eb3ce8700a780f76034b042897ac032197fdbc6f0",
        "weight_tensor_count": 13
      },
      {
        "file": "model-plefp8-00005.safetensors",
        "header_size": 2222,
        "header_sha256": "0e31233d36c6242b50879dc98732ae4f5ba8686d89399dfbaff214c2df3f9cef",
        "weight_tensor_count": 13
      },
      {
        "file": "model-plefp8-00006.safetensors",
        "header_size": 2222,
        "header_sha256": "d1d66c9d1dd3d8e6c0b4705d15699167a02539d0e85fbde209c6f9c9fce3fb5e",
        "weight_tensor_count": 13
      },
      {
        "file": "model-plefp8-00007.safetensors",
        "header_size": 2222,
        "header_sha256": "9afb413b2606c63ebd419e92fd36b40922397856ae437a9df63e6c1bfa545d9b",
        "weight_tensor_count": 13
      },
      {
        "file": "model-plefp8-00008.safetensors",
        "header_size": 2222,
        "header_sha256": "939d0f5043f1459b79b1d3774c15778d18f86ce0bcb42111df00f6c6dadc74ea",
        "weight_tensor_count": 13
      },
      {
        "file": "model-plefp8-00009.safetensors",
        "header_size": 2033,
        "header_sha256": "e0df50b7e0d82ccad1270d0bcb99a862f4b4591942ccea156054a8420ac001cc",
        "weight_tensor_count": 11
      }
    ],
    "bf16_header": {
      "file": "model-00016-of-00017.safetensors",
      "header_size": 20728,
      "header_sha256": "2d42b89d8239a44849504e4ee02db22d999da6c7ea1f5a47e570d73f773332e0",
      "weight_tensor_count": 128
    },
    "observed_all_128_shapes_dtypes": true,
    "observed_scale_value": true,
    "kaggle_mirror_verified": false
  },
  "upstream_nonple_lfs_metadata": {
    "identical_shards": [
      {
        "fp8_file": "model-00001-of-00017.safetensors",
        "bf16_file": "model-00001-of-00017.safetensors",
        "sha256": "70734bddc4db7476d4ba03a86f808290e88488d39bf513c639ad3957768692de",
        "size": 2899311784
      },
      {
        "fp8_file": "model-00003-of-00017.safetensors",
        "bf16_file": "model-00003-of-00017.safetensors",
        "sha256": "0ecb3e52cad98e4c3c042432f3db2f12ca6c41ea6844a964f29e66f673fb5be5",
        "size": 5370548192
      },
      {
        "fp8_file": "model-00004-of-00017.safetensors",
        "bf16_file": "model-00004-of-00017.safetensors",
        "sha256": "2865e0756b9a3cc0e045eeec349d3e495d1630476cc5355ea0da6a272092d9b0",
        "size": 5370910416
      },
      {
        "fp8_file": "model-00005-of-00017.safetensors",
        "bf16_file": "model-00005-of-00017.safetensors",
        "sha256": "5f0b9642174a7bfbfa57c6fda7917effeb3afb23c2377cfc5f8f9e29222dd048",
        "size": 5370140704
      },
      {
        "fp8_file": "model-00006-of-00017.safetensors",
        "bf16_file": "model-00006-of-00017.safetensors",
        "sha256": "0d06de9de63a1cdabd39a6c11ea1b6431b24be17c4a51fe4db236ea77c33d0b9",
        "size": 5370205184
      },
      {
        "fp8_file": "model-00007-of-00017.safetensors",
        "bf16_file": "model-00007-of-00017.safetensors",
        "sha256": "a632e777190fac1ee19b8706925862af7b4f3ce44e58b40f01a79c0ab7697e96",
        "size": 5370927288
      },
      {
        "fp8_file": "model-00008-of-00017.safetensors",
        "bf16_file": "model-00008-of-00017.safetensors",
        "sha256": "cfc3279b2981d2b6c319b6be03d33091c0dea1b94d194705450b4cece4df6a7d",
        "size": 5344445352
      },
      {
        "fp8_file": "model-00009-of-00017.safetensors",
        "bf16_file": "model-00009-of-00017.safetensors",
        "sha256": "f3f8390dc32f3a2ea3980fa1a44cc37d02394c40388bd4688b16779431791327",
        "size": 5370355648
      },
      {
        "fp8_file": "model-00010-of-00017.safetensors",
        "bf16_file": "model-00010-of-00017.safetensors",
        "sha256": "48f9d03732bc89e177b946da67702e3eeec55d185a4c795dfebfecf380014d42",
        "size": 5370927288
      },
      {
        "fp8_file": "model-00011-of-00017.safetensors",
        "bf16_file": "model-00011-of-00017.safetensors",
        "sha256": "8592f8599a883e1a57f4010b3cfa7516e8f1c5777d21a62f9b0e20e06e56840c",
        "size": 5324173744
      },
      {
        "fp8_file": "model-00012-of-00017.safetensors",
        "bf16_file": "model-00012-of-00017.safetensors",
        "sha256": "6d58f6a6ac680e6cde7e061025a55cac650ec32f00a7ae1fb6aa95839971ba40",
        "size": 5370438520
      },
      {
        "fp8_file": "model-00013-of-00017.safetensors",
        "bf16_file": "model-00013-of-00017.safetensors",
        "sha256": "901b9e3f509df9a051d99ad5a9815339b68b4e78006805175e7c1f75423883c0",
        "size": 5370927288
      },
      {
        "fp8_file": "model-00014-of-00017.safetensors",
        "bf16_file": "model-00014-of-00017.safetensors",
        "sha256": "a77beb75499e376d0de54232d510f533eaf730149c4d9f69e5b327850b3da66b",
        "size": 5342689856
      },
      {
        "fp8_file": "model-00015-of-00017.safetensors",
        "bf16_file": "model-00015-of-00017.safetensors",
        "sha256": "f457d062094bda4c6bd486fe373ae56d04f2b205ef083e95cfdee9400da86d2c",
        "size": 5050474736
      },
      {
        "fp8_file": "model-00017-of-00017.safetensors",
        "bf16_file": "model-00017-of-00017.safetensors",
        "sha256": "fdbe0ca6b1bfab1dece276c48efadbc7d24ebb4c2619497c19db47b400f31782",
        "size": 1284526608
      }
    ],
    "total_identical_file_bytes": 73581002608,
    "locally_full_file_hashed": false,
    "not_a_kaggle_mirror_integrity_proof": true
  },
  "runtime": {
    "repo": "https://github.com/jpezzulli/sglang-rtxpro6000",
    "revision": "d00d88efc8d6281b12be4f4073126aec95038c55",
    "wheelhouse_kaggle_source": "dfranzen/pennyroyal-v253",
    "ple_offload_embedding": true,
    "global_compute_dtype": "bfloat16",
    "fusion_default": true,
    "fusion_fast_path_guards": [
      "SM120",
      "TP1",
      "FP8 pinned contiguous table",
      "head width160",
      "target verify context count4or16"
    ],
    "runtime_missing_scale_silently_defaults_to_one": true,
    "runtime_does_not_check_positive_finite_scale": true,
    "frozen_other_settings": "AutoRound target; independent INT4/group32 draft; KV/Mamba dtypes; context; concurrency; sampling; caches; schedule."
  },
  "sources_sha256": {
    "009-effective-server-args.json": "7ebb68ae6e67bde2d369e6473a972e1e55b58802108ac28fbe2060425d1402f6",
    "fetch_metadata.py": "f840c945f6f92d0b61d107c36ebcece2795f16d20ecf26753135790af5c701aa",
    "comparison.json": "146039b7a5c1c6236cc0fd0f76ab1240f3f133ac9d624e2811db7366cb8c2dc6",
    "intel-qwen3.8-flash-next-w4a16-autoround-kaggle-files.json": "4a70dd4d1b6ad562b9f1e1991a20f8064d7757594b931fcbfd77f4b521b707fd",
    "analyze_fp8.py": "78f7b787ac2f0d679c8b8b9d58b2d62083774d0b667d1a279a06ce343873ad16",
    "009-serving-cell.py": "4f20381acf0dba40e150b5ba99703d19098007724f9623a04c12518cb2721a7c",
    "fetch_serving.py": "33554ef8df22494b74db330c3c566b07e1a9a517da76f16d5a121c2a8b343d89",
    "albucino-qwen3.8-flash-next-w4a16-fp8ple-kaggle-files.json": "1031e6a103ed7e0558b2510aa13834f7ee952a149f2920be4002137b3af2bdce",
    "Intel/model-api.json": "d32494cf287b0a7ef5551976773cd5023da3bbd18e68b8ede47c9a3b788ec214",
    "Intel/config.json": "0fabfa21fab8bfe69f02234f7ae8df4ed91b785fca2e22011c1230f6a07e5329",
    "Intel/README.md": "00798492330afbb7285fd9c0deee4fc7ac01bbc197eb49d1d5c868ceafe54293",
    "Intel/quantization_config.json": "2371a9d0bed1ff619622ac57aa0e6e5b39688bb834b48d761e90d5834f4cab4b",
    "Intel/model.safetensors.index.json": "35e503b758cf5441886042d315c48dfba70f947611816cbb90e02d4d5c5e58ff",
    "pennyroyal/README.md": "af19a7ab6fe51d2d9f5f091d1a84ff81e7e9353bc11898013bd2bbc38f2fcb2d",
    "pennyroyal/python/sglang/kernels/ops/qwen4_ple.py": "7f64e896c1f00f46ab6c4cd8c5c90cedc528555d1cc7401e9c7a5a3a31b87f16",
    "pennyroyal/python/sglang/srt/environ.py": "84755bded97394a18537d8e329c3d05c15f7ec4aa8a801bd9971f34b4f11d7ed",
    "pennyroyal/python/sglang/srt/models/qwen4_exp.py": "35a1785ce4c287821f00f1518b3880ab6838ef58d42fd51ece86d8098b2ed666",
    "pennyroyal/python/sglang/srt/model_loader/loader.py": "1016b2c6c2d7eb50ef25e0a731e20096e4b79239bccb6fea26aa59d3f29bf8fe",
    "albucino/model-api.json": "d8ae1760db67b760da8d67327a13a702e4b45e7b678c9ccbf15d7c03af911c50",
    "albucino/config.json": "2925a0d0989384e0ef7973a14c97ba9c8dfc82c4ca3684d2e83cf4f57db6c172",
    "albucino/README.md": "3c1ecf16c50cdca78d208dedc5a70cc974e154a83e2e5ea62f6df072cdd37497",
    "albucino/model.safetensors.index.json": "eb08a649acf89aa2d0c5c1858704cba8e86cdf2b93dec8cf9a89024292a3cc5a",
    "albucino/hybrid_sources.json": "1a4d1bd0f8c60d6735013d8eadc46262c0f7d7872724c22e8909085611afa565"
  },
  "primary_format_sources": [
    {
      "url": "https://huggingface.co/docs/safetensors/index",
      "accessed_at_utc": "2026-10-18",
      "scope": "safe_open.get_slice partial tensor access"
    },
    {
      "url": "https://github.com/huggingface/safetensors/blob/main/README.md",
      "accessed_at_utc": "2026-10-18",
      "scope": "8-byte little-endian length; JSON descriptors; relative offsets. Dynamic documentation is not a runtime pin."
    }
  ],
  "expected_normalized_config_sha256": "cd12e205b532c8558273dc61b3fbb59e75ca6949bdda41bc0eb6f85d3610231c",
  "validation": {
    "performed": "temporary synthetic safetensors, CPU only, no network or model code",
    "valid_fixture": "passed",
    "negative_cases_rejected": [
      "scale differs",
      "non-PLE sampled bytes differ",
      "FP8 NaN",
      "FP8 numerical bound exceeded",
      "PLE shard missing",
      "normalized frozen config hash mismatch",
      "asymmetric draft"
    ],
    "cli": "python -m arc3_research.ple_weights succeeded on temporary valid fixture; exit0 and passed=true"
  }
}


In [ ]:
# Static JSON data only; never eval/exec model-generated content.
import base64, lzma, hashlib
suite_bytes_019 = lzma.decompress(base64.b64decode('...'))
assert hashlib.sha256(suite_bytes_019).hexdigest() == 'a2cbdcc02cb48ae73c64016cc45b2af0c0e54315efeedac328d1b9d473f8792c'
Path('/kaggle/working/019-code/suite.json').write_bytes(suite_bytes_019)
del suite_bytes_019


In [ ]:
%%writefile /kaggle/working/019-code/launch_server.py
# Derived from Daniel Franzen Milestone 2 v3 serving cell (Apache-2.0 harness).
# CPU metadata only until the original pinned launcher starts its owned server.
import hashlib, json, os, signal, sys, time
from pathlib import Path
from arc3_research.session_runner import stop_group
from arc3_research.ple_session import save

PARAMETERS = json.loads(Path(sys.argv[1]).read_text())
MODEL_DIR, DRAFT_MODEL_DIR = PARAMETERS["model"], PARAMETERS["draft"]
WHEELHOUSE_DIR, WORKING_DIR = PARAMETERS["wheelhouse"], Path(PARAMETERS["working_dir"])
SERVED_MODEL_NAME, SERVED_MODEL_PORT = "flashnext", PARAMETERS["port"]
NOTEBOOK_START_TIME = time.time()
SERVER_STARTUP_TIMEOUT = PARAMETERS["startup_limit_seconds"]
proc = None
RUNTIME_AUDIT = {}

def _audit_installed_runtime():
    expected = PARAMETERS["runtime_source_sha256"]
    roots = list(Path(VENV).glob("lib/python*/site-packages"))
    for relative, digest in expected.items():
        paths = [r / relative for r in roots if (r / relative).is_file()]
        if len(paths) != 1:
            raise RuntimeError("Pinned runtime source absent or ambiguous: " + relative)
        observed = hashlib.sha256(paths[0].read_bytes()).hexdigest()
        RUNTIME_AUDIT[relative] = {"expected": digest, "observed": observed, "matches": observed == digest}
        if observed != digest:
            save(WORKING_DIR / "runtime-source-check.json", RUNTIME_AUDIT)
            raise RuntimeError("Installed serving source differs from audited pinned source: " + relative)
    save(WORKING_DIR / "runtime-source-check.json", RUNTIME_AUDIT)

def _write_state(status, **extra):
    metadata_files = ["config.json", "model.safetensors.index.json", "tokenizer.json", "chat_template.jinja"]
    save(WORKING_DIR / "launcher-state.json", {
        "status": status, "epoch": time.time(), "launcher_pid": os.getpid(),
        "server_pid": proc.pid if proc is not None else None,
        "args": globals().get("args"), "runtime_source_check": RUNTIME_AUDIT,
        "target_metadata_sha256": {n: hashlib.sha256(Path(MODEL_DIR, n).read_bytes()).hexdigest()
            for n in metadata_files if Path(MODEL_DIR, n).is_file()},
        "environment": {k: v for k, v in globals().get("env", {}).items()
            if k.startswith("SGLANG_") or k in {"OMP_NUM_THREADS", "MKL_NUM_THREADS", "CUDA_VISIBLE_DEVICES"}},
        **extra})

def _terminate(signum, frame):
    raise SystemExit(128 + signum)

signal.signal(signal.SIGTERM, _terminate)
signal.signal(signal.SIGINT, _terminate)
try:
    import os
    import threading
    import time

    from concurrent.futures import as_completed, ThreadPoolExecutor

    def precache(*paths, times=1, delay=0, threads=16) -> None:
        time.sleep(delay)
        print(f'Precaching: Started...')
        start_time = time.time()

        CHUNK_SIZE = 32 * 1024 * 1024  # 128 MB

        chunks = []  # list of (path, offset, length)
        total_size = 0
        file_count = 0

        def _add_file(i: int, path: str) -> None:
            nonlocal total_size, file_count
            size = os.path.getsize(path)
            total_size += size
            file_count += 1
            for offset in range(0, max(size, 1), CHUNK_SIZE):
                length = min(CHUNK_SIZE, size - offset)
                chunks.append((i, path, offset, length))

        for i, path in enumerate(paths):
            try:
                if os.path.isfile(path):
                    _add_file(i, path)
                    continue
                for root, _, files in os.walk(path):
                    for file_name in files:
                        file_path = os.path.join(root, file_name)
                        if os.path.isfile(file_path):
                            _add_file(i, file_path)
            except: pass

        chunks.sort()

        _local = threading.local()

        def _read_chunk(chunk: tuple) -> None:
            try:
                buf = _local.buf
            except AttributeError:
                _local.buf = buf = bytearray(CHUNK_SIZE)
            i, path, offset, _ = chunk
            try:
                with open(path, 'rb') as f:
                    f.seek(offset)
                    remaining = CHUNK_SIZE
                    for i in range(1024):
                        n = f.readinto(buf)
                        if not n:
                            break
                        remaining -= n
                        if remaining <= 0: break
            except: pass

        with ThreadPoolExecutor(max_workers=threads) as pool:
            for _ in range(times):
                list(pool.map(_read_chunk, chunks))

        elapsed = time.time() - start_time
        print(f'Precaching: loaded {file_count} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

    precache_wheels_thread = threading.Thread(target=precache, args=(WHEELHOUSE_DIR,))
    precache_wheels_thread.start()
    precache_wheels_thread.join()

    precache_model_thread = threading.Thread(target=precache, args=(MODEL_DIR, DRAFT_MODEL_DIR), kwargs={'delay': 60, 'threads': 3})
    # Paste this entire file into the launcher cell.
    # Required notebook variables: MODEL_DIR, DRAFT_MODEL_DIR, WHEELHOUSE_DIR,
    # WORKING_DIR, SERVED_MODEL_NAME, SERVED_MODEL_PORT.
    # MODEL_DIR: Intel AutoRound INT4 target (BF16 PLE or albucino FP8 PLE).
    # DRAFT_MODEL_DIR: downloaded albucino runtime/mtp-int4-g32, or its dataset root.
    import os, sys, glob, time, shutil, subprocess, urllib.request, urllib.error
    import copy, hashlib, json, re, shlex, socket, zipfile
    from pathlib import Path

    PREFIX = "/tmp/sgl-intel"       # Separate from the previous fork's /tmp/sgl venv.
    LOG = str(Path(WORKING_DIR) / "serve.log")
    CFG = dict(
        CTX=(116+12+8)*1024,
        MEMFRAC=0.96,
        MAXREQ=10,
        CUDAGRAPH_MAXBS=10,
        MAMBA_CACHE=60,
        CHUNK=8192,
        MAX_PREFILL=16384,
        SPEC=True,
        FRSPEC=True,
        AUTOTUNE=True,
        # Preserve the original Kaggle checkpoint prefetch/cache behavior.
        PREFETCH_CHECKPOINTS=True,
        DROP_CACHE_AFTER_LOAD=True,
        LINEAR_BACKEND="flashinfer",
        KVDTYPE="fp8_e4m3",
        SSM_DTYPE="bfloat16",
        MAMBA_RADIX="extra_buffer",
        GDN_MTP_CACHE_MODE="none",
        SERVED_NAME=SERVED_MODEL_NAME,
        SPEC_ACCEPT_SINGLE=1.0,
        SPEC_ACCEPT_ACC=1.0,
        SPEC_STEPS=3,
    )
    VENV = f"{PREFIX}/venv"
    PYTHON = f"{VENV}/bin/python"
    SGLANG = f"{VENV}/bin/sglang"
    TOKEN_MAP_SHA = "becfa41d394b86c26c632bea8f3c6ea64bbb76d7b238d8673c06afae21269f25"
    TOKENIZER_SHA = "06b9509352d2af50381ab2247e083b80d32d5c0aba91c272ca9ff729b6a0e523"
    EXPERT_TARGET = r"re:^mtp\.layers\.0\.mlp\.experts\.[0-9]+\.(gate_proj|up_proj|down_proj)$"
    DENSE_IGNORE = r"re:^(?!mtp\.layers\.0\.mlp\.experts(?:\.|$)).*"


    def run(cmd, *, check=True, env=None):
        cmd = list(map(str, cmd))
        print("$", shlex.join(cmd))
        result = subprocess.run(cmd, text=True, capture_output=True, env=env)
        if result.stdout.strip(): print(result.stdout.strip()[-6000:])
        if result.stderr.strip() and result.returncode: print(result.stderr.strip()[-6000:])
        if check and result.returncode:
            raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(cmd)}")
        return result


    def find_unique(root, name, *, directory=False, required=True):
        root = Path(root)
        direct = root if root.name == name else root / name
        valid = lambda p: p.is_dir() if directory else p.is_file()
        if valid(direct): return direct
        hits = sorted(p for p in root.rglob(name) if valid(p))
        if len(hits) > 1:
            raise RuntimeError(f"Multiple {name} matches under {root}; use a more specific directory: {hits}")
        if hits: return hits[0]
        if required: raise FileNotFoundError(f"{name} not found under {root}")
        return None


    def sha256(path):
        h = hashlib.sha256()
        with Path(path).open("rb") as f:
            for block in iter(lambda: f.read(8*1024*1024), b""): h.update(block)
        return h.hexdigest()


    def indexed_shards(root):
        root = Path(root)
        index = json.loads((root / "model.safetensors.index.json").read_text())
        names = sorted(set(index["weight_map"].values()))
        if not names: raise RuntimeError(f"Empty weight index: {root}")
        for name in names:
            rel = Path(name)
            if rel.is_absolute() or ".." in rel.parts:
                raise RuntimeError(f"Invalid shard path: {name}")
            p = root / rel
            if not p.is_file() or p.stat().st_size < 8:
                raise RuntimeError(f"Missing/empty shard: {p}")
        return index, names


    def prepare_draft_view(target, source_root, work_root):
        """Adapt metadata only; link original dataset weights without copying them."""
        target, source_root, work_root = map(Path, (target, source_root, work_root))
        configs = [source_root / "config.json"] if (source_root / "config.json").is_file() else []
        configs += sorted(p for p in source_root.rglob("config.json") if p not in configs)
        candidates = []
        for path in configs:
            cfg = json.loads(path.read_text())
            q = cfg.get("quantization_config", {})
            if q.get("quant_method") == "compressed-tensors" and "mtp_routed_experts" in q.get("config_groups", {}):
                candidates.append((path.parent, cfg))
        if len(candidates) != 1:
            raise RuntimeError(f"Expected one INT4 g32 MTP checkpoint under DRAFT_MODEL_DIR={source_root}; found {len(candidates)}")
        source, cfg = candidates[0]
        index, shards = indexed_shards(source)
        if not any(k.startswith("mtp.layers.0.mlp.experts.") for k in index["weight_map"]):
            raise RuntimeError("Draft index does not contain the expected MTP experts")
        cfg = copy.deepcopy(cfg)
        q = cfg["quantization_config"]
        group = q["config_groups"]["mtp_routed_experts"]
        w = group["weights"]
        if (w["num_bits"], w["group_size"], w["symmetric"]) != (4, 32, True):
            raise RuntimeError("Expected symmetric INT4 group32 draft experts")
        if group["targets"] not in (["RoutedExperts"], [EXPERT_TARGET]):
            raise RuntimeError(f"Unexpected draft target rules: {group['targets']}")
        if q.get("ignore", []) not in ([], [DENSE_IGNORE]):
            raise RuntimeError("Unexpected draft ignore rules")
        group["targets"] = [EXPERT_TARGET]
        q["ignore"] = [DENSE_IGNORE]
        # Content/path-specific directory makes notebook reruns idempotent without
        # overwriting an unrelated view. We never modify the read-only datasets.
        identity = json.dumps({"source": str(source.resolve()), "target": str(target.resolve()), "config": cfg}, sort_keys=True)
        suffix = hashlib.sha256(identity.encode()).hexdigest()[:16]
        view = work_root / f"draft-view-{suffix}"
        view.mkdir(parents=True, exist_ok=True)
        config_path = view / "config.json"
        config_text = json.dumps(cfg, indent=2) + "\n"
        if config_path.exists() or config_path.is_symlink():
            if config_path.is_symlink() or config_path.read_text() != config_text:
                raise RuntimeError(f"Unexpected existing draft-view config: {config_path}")
        else:
            config_path.write_text(config_text)
        links = {name: source / name for name in shards}
        links["model.safetensors.index.json"] = source / "model.safetensors.index.json"
        for name in ("tokenizer.json", "tokenizer_config.json", "chat_template.jinja", "preprocessor_config.json", "generation_config.json"):
            if (target / name).is_file(): links[name] = target / name
        for name, src in links.items():
            dest = view / name
            dest.parent.mkdir(parents=True, exist_ok=True)
            if dest.exists() or dest.is_symlink():
                if not dest.is_symlink() or dest.resolve() != src.resolve():
                    raise RuntimeError(f"Unexpected existing draft-view file: {dest}")
            else:
                dest.symlink_to(src.resolve())
        print(f"Draft source: {source}\nSGLang draft view: {view} ({len(shards)} linked weight shards)")
        return view


    # ---- validate notebook inputs / identify the offline bundle ----
    MODEL_DIR = str(Path(MODEL_DIR).expanduser().resolve())
    assert Path(MODEL_DIR).is_dir(), f"model dir not found: {MODEL_DIR}"
    assert CFG["CUDAGRAPH_MAXBS"] >= CFG["MAXREQ"], "CUDAGRAPH_MAXBS must cover MAXREQ"
    assert CFG["CHUNK"] > 0 and CFG["CHUNK"] % 64 == 0
    # Refuse to start a second server on an occupied port; do not kill other jobs.
    with socket.socket() as probe:
        if probe.connect_ex(("127.0.0.1", int(SERVED_MODEL_PORT))) == 0:
            raise RuntimeError(f"Port {SERVED_MODEL_PORT} is occupied; stop the existing server before rerunning")
    Path(PREFIX).mkdir(parents=True, exist_ok=True)
    Path(LOG).parent.mkdir(parents=True, exist_ok=True)
    WHEELS = find_unique(WHEELHOUSE_DIR, "wheels", directory=True)
    LOCK = find_unique(WHEELHOUSE_DIR, "requirements.lock")
    sglang_wheels = sorted(WHEELS.glob("sglang-*.whl"))
    assert len(sglang_wheels) == 1, f"Expected one SGLang wheel, found {sglang_wheels}"
    wheel = sglang_wheels[0]
    install_marker = Path(PREFIX) / "installed-bundle.txt"

    # ---- one-time offline install, isolated from the old mratsim environment ----
    install_env = dict(os.environ, UV_OFFLINE="1", UV_PYTHON_DOWNLOADS="never", PYTHONNOUSERSITE="1")
    install_env.pop("PYTHONPATH", None)
    if not Path(SGLANG).exists() or not install_marker.is_file():
        print("=== installing Pennyroyal (offline) ===")
        # Bootstrap uv directly from its wheel; no global pip changes/ensurepip.
        uv_wheels = sorted(WHEELS.glob("uv-*.whl"))
        assert len(uv_wheels) == 1, "Expected one bundled uv wheel"
        uv_binary = Path(PREFIX) / "bin/uv"
        uv_binary.parent.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(uv_wheels[0]) as archive:
            members = [n for n in archive.namelist() if Path(n).name == "uv" and not n.endswith("/")]
            assert len(members) == 1, "Cannot identify the Linux uv executable in its wheel"
            binary = archive.read(members[0])
            assert binary.startswith(b"\x7fELF"), "Bundled uv executable is not a Linux ELF binary"
            uv_binary.write_bytes(binary)
        uv_binary.chmod(0o755)
        uv = [str(uv_binary)]
        if not Path(PYTHON).exists():
            run(uv + ["venv", "--python", sys.executable, VENV], env=install_env)
        install = uv + ["pip", "install", "--python", PYTHON, "--no-index", "--find-links", str(WHEELS)]
        run(install + ["-r", str(LOCK)], env=install_env)
        run(install + ["--reinstall", "--no-deps", str(wheel)], env=install_env)
        install_marker.write_text('install')
    else:
        print(f"Matching offline bundle already installed at {VENV}")

    # ---- bundled CUDA toolkit and linker symlinks ----
    cu13 = sorted(Path(VENV).glob("lib/python*/site-packages/nvidia/cu13"))
    if not cu13 or not (cu13[0] / "bin/nvcc").is_file():
        raise RuntimeError("Bundled CUDA 13 nvcc missing; check that the complete Pennyroyal wheelhouse was uploaded")
    CUDA_HOME = str(cu13[0])
    libdir = Path(CUDA_HOME) / "lib"
    lib64 = Path(CUDA_HOME) / "lib64"
    if not lib64.exists() and not lib64.is_symlink(): lib64.symlink_to("lib")
    for so in sorted(libdir.glob("*.so.*")):
        link = libdir / re.sub(r"\.so\..*$", ".so", so.name)
        if not link.exists() and not link.is_symlink(): link.symlink_to(so.name)
    assert (libdir / "libcudart.so").exists(), "libcudart.so missing: JIT linker cannot find CUDA runtime"
    if not (libdir / "libcuda.so").exists():
        candidates = []
        for directory in ("/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
            candidates += sorted(p for p in Path(directory).glob("libcuda.so*") if p.is_file())
        if not candidates:
            code = 'import torch; torch.cuda.init(); print(open("/proc/self/maps").read())'
            maps = run([PYTHON, "-c", code], env=install_env).stdout
            candidates = [Path(p) for p in re.findall(r"(/[^\s]*libcuda\.so[^\s]*)", maps)]
        if not candidates: raise RuntimeError("GPU driver libcuda.so not found")
        link = libdir / "libcuda.so"
        if link.is_symlink(): link.unlink()
        link.symlink_to(candidates[0].resolve())
    CXX = next((c for c in ("g++-15", "g++-14", "g++-13", "g++-12", "g++-11", "g++") if shutil.which(c)), None)
    assert CXX, "Host C++ compiler not found"
    CC = CXX.replace("g++", "gcc")
    assert shutil.which(CC), f"Matching C compiler missing: {CC}"

    # ---- runtime environment ----
    env = dict(os.environ)
    env.pop("PYTHONPATH", None)
    cache = f"{PREFIX}/cache"
    for name in ("huggingface", "torch", "torchinductor", "triton", "flashinfer", "cuda", "sglang/jit"):
        Path(cache, name).mkdir(parents=True, exist_ok=True)
    env.update({
        "PYTHONNOUSERSITE": "1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
        "HF_DATASETS_OFFLINE": "1", "UV_OFFLINE": "1", "UV_PYTHON_DOWNLOADS": "never",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "CUDA_HOME": CUDA_HOME, "CUDACXX": f"{CUDA_HOME}/bin/nvcc",
        "PATH": f"{CUDA_HOME}/bin:{VENV}/bin:" + env.get("PATH", ""),
        "LD_LIBRARY_PATH": f"{CUDA_HOME}/lib64:{CUDA_HOME}/lib:" + env.get("LD_LIBRARY_PATH", ""),
        "CC": CC, "CXX": CXX, "CUDAHOSTCXX": CXX,
        "TORCH_CUDA_ARCH_LIST": "12.0",
        "HF_HOME": f"{cache}/huggingface", "XDG_CACHE_HOME": cache,
        "TORCH_HOME": f"{cache}/torch", "CUDA_CACHE_PATH": f"{cache}/cuda",
        "TORCHINDUCTOR_CACHE_DIR": f"{cache}/torchinductor", "TRITON_CACHE_DIR": f"{cache}/triton",
        "FLASHINFER_WORKSPACE_BASE": f"{cache}/flashinfer",
        "SGLANG_CACHE_DIR": f"{cache}/sglang", "SGLANG_JIT_CACHE_DIR": f"{cache}/sglang/jit",
        "SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN": "1",
        "OMP_NUM_THREADS": "8", "MKL_NUM_THREADS": "8", "TOKENIZERS_PARALLELISM": "false",
        "MAX_JOBS": "8", "CMAKE_BUILD_PARALLEL_LEVEL": "8",
        "FLASHINFER_NINJA_JOBS": "8", "FLASHINFER_NVCC_THREADS": "2", "TORCHINDUCTOR_COMPILE_THREADS": "8",
        "SGLANG_ENABLE_SM120_LOWM_BF16_GEMM": "1", "SGLANG_SM120_ONLINE_MXFP8": "0",
        "SGLANG_SM120_LOWM_FP8_WEIGHT": "0", "SGLANG_SM120_LM_HEAD_FP8": "0",
        "SGLANG_MAMBA_CONV_DTYPE": "bfloat16", "SGLANG_NUMA_BIND_V2": "false",
        "SGLANG_MM_PREPROCESS_DEVICE": "cpu", "NUMPY_MADVISE_HUGEPAGE": "0",
    })
    run([str(Path(CUDA_HOME) / "bin/nvcc"), "--version"], env=env)
    print("Host compiler:", CXX)
    # Report the optional later caching patch. This cell does not modify wheel code.
    run([PYTHON, "-c", '''import pathlib, sglang, torch
    print("sglang", sglang.__version__, "| torch", torch.__version__)
    assert torch.cuda.is_available(), "CUDA unavailable"
    print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))
    '''], env=env)

    _audit_installed_runtime()

    # ---- validate target and create the read-only draft's SGLang view ----
    target_cfg = json.loads(Path(MODEL_DIR, "config.json").read_text())
    q = target_cfg.get("quantization_config", {})
    assert q.get("quant_method") == "auto-round" and q.get("bits") == 4, "MODEL_DIR must be the Intel/albucino AutoRound INT4 checkpoint"
    _, target_shards = indexed_shards(MODEL_DIR)
    print("Target:", MODEL_DIR, "| shards:", len(target_shards), "| PLE dtype:", target_cfg.get("text_config", {}).get("ple_embedding_dtype", "checkpoint default (typically BF16)"))
    DRAFT_VIEW = None
    if CFG["SPEC"]:
        assert Path(DRAFT_MODEL_DIR).is_dir(), f"draft dir not found: {DRAFT_MODEL_DIR}"
        DRAFT_VIEW = prepare_draft_view(MODEL_DIR, DRAFT_MODEL_DIR, PREFIX)

    # ---- assemble Pennyroyal / AutoRound launch arguments ----
    graph_bs = sorted({1, 2, 4, 7, 8, 9, 10, CFG["MAXREQ"], CFG["CUDAGRAPH_MAXBS"]})
    graph_bs = [n for n in graph_bs if n <= CFG["CUDAGRAPH_MAXBS"]]
    args = [SGLANG, "serve", "--model-path", MODEL_DIR, "--load-format", "safetensors",
        "--model-loader-extra-config", '{"enable_multithread_load":false}',
        "--served-model-name", str(CFG["SERVED_NAME"]), "--host", "127.0.0.1",
        "--port", str(SERVED_MODEL_PORT), "--tensor-parallel-size", "1",
        "--dtype", "bfloat16", "--quantization", "auto-round",
        "--kv-cache-dtype", CFG["KVDTYPE"], "--mem-fraction-static", str(CFG["MEMFRAC"]),
        "--context-length", str(CFG["CTX"]), "--page-size", "64",
        "--max-running-requests", str(CFG["MAXREQ"]), "--chunked-prefill-size", str(CFG["CHUNK"]),
        "--max-prefill-tokens", str(CFG["MAX_PREFILL"]),
        "--cuda-graph-max-bs-decode", str(CFG["CUDAGRAPH_MAXBS"]),
        "--cuda-graph-bs-decode", *map(str, graph_bs),
        "--mamba-ssm-dtype", CFG["SSM_DTYPE"], "--max-mamba-cache-size", str(CFG["MAMBA_CACHE"]),
        "--mamba-radix-cache-strategy", CFG["MAMBA_RADIX"], "--mamba-track-interval", "64",
        "--mamba-backend", CFG["LINEAR_BACKEND"],
        "--linear-attn-decode-backend", CFG["LINEAR_BACKEND"],
        "--linear-attn-prefill-backend", CFG["LINEAR_BACKEND"],
        "--moe-runner-backend", "auto", "--ple-offload-embedding", "--trust-remote-code",
        "--mm-feature-transport", "cpu", "--image-processor-backend", "pil",
        "--reasoning-parser", "qwen3", "--tool-call-parser", "qwen3_coder",
        "--default-chat-template-kwargs", '{"preserve_thinking":true}',
        "--watchdog-timeout", "1800", "--schedule-policy", "lpm", "--warmups", "structured_output",
        "--enable-cache-report", "--enable-metrics", "--enable-request-time-stats-logging"]
    if CFG["PREFETCH_CHECKPOINTS"]: args += ["--weight-loader-prefetch-checkpoints"]
    if CFG["DROP_CACHE_AFTER_LOAD"]: args += ["--weight-loader-drop-cache-after-load"]
    tmpl = Path(MODEL_DIR, "chat_template.jinja")
    if tmpl.is_file(): args += ["--chat-template", str(tmpl)]
    if not CFG["AUTOTUNE"]: args += ["--disable-flashinfer-autotune"]
    if CFG["GDN_MTP_CACHE_MODE"]: args += ["--gdn-mtp-cache-mode", CFG["GDN_MTP_CACHE_MODE"]]
    if CFG["SPEC"]:
        args += ["--speculative-algorithm", "NEXTN", "--speculative-num-steps", str(CFG["SPEC_STEPS"]),
            "--speculative-eagle-topk", "1", "--speculative-num-draft-tokens", str(CFG["SPEC_STEPS"]+1),
            "--speculative-draft-model-path", str(DRAFT_VIEW),
            "--speculative-draft-model-quantization", "compressed-tensors", "--speculative-moe-runner-backend", "auto",
            "--speculative-draft-kv-cache-dtype", CFG["KVDTYPE"],
            "--speculative-accept-threshold-single", str(CFG["SPEC_ACCEPT_SINGLE"]),
            "--speculative-accept-threshold-acc", str(CFG["SPEC_ACCEPT_ACC"])]
        if CFG["FRSPEC"]:
            tok = find_unique(WHEELHOUSE_DIR, "hot_tokens_64k.pt", required=False)
            if tok is None: tok = find_unique(WHEELHOUSE_DIR, "flash-next-64k.pt")
            assert sha256(tok) == TOKEN_MAP_SHA, "Wrong FR-Spec map; use the map from the pinned Pennyroyal bundle"
            assert sha256(Path(MODEL_DIR, "tokenizer.json")) == TOKENIZER_SHA, "Tokenizer differs from the FR-Spec map; set CFG['FRSPEC']=False"
            args += ["--speculative-token-map", str(tok)]

    # ---- launch detached and wait for health ----
    args += ["--random-seed", str(PARAMETERS["seed"])]
    precache_model_thread.start()
    print("\n" + shlex.join(args) + "\n")
    log_offset = Path(LOG).stat().st_size if Path(LOG).exists() else 0
    with open(LOG, "ab", buffering=0) as logf:
        proc = subprocess.Popen(args, env=env, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True)
    print(f"pid {proc.pid} -> {LOG}")
    _write_state("loading")

    def show_log_tail():
        with open(LOG, "rb") as f:
            f.seek(0, 2)
            f.seek(max(log_offset, f.tell()-20000))
            print(f.read().decode(errors="replace"))

    url, t0 = f"http://127.0.0.1:{SERVED_MODEL_PORT}/health", time.time()
    last_report = -30
    while time.time() - NOTEBOOK_START_TIME < SERVER_STARTUP_TIMEOUT:
        if proc.poll() is not None:
            show_log_tail()
            break
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    print(f"\nREADY after {int(time.time()-t0)}s -> {url.replace('/health','/v1')}")
                    break
        except Exception:
            pass
        elapsed = int(time.time()-t0)
        if elapsed-last_report >= 30:
            print(f"  {elapsed}s ...", flush=True); last_report=elapsed
        time.sleep(3)
    else:
        print(f"\nDEADLINE at {int(time.time()-NOTEBOOK_START_TIME)}s from notebook start; "
              f"releasing the benchmark with the server still loading")
    if proc.poll() is not None:
        raise RuntimeError("Model server exited during startup")
    with urllib.request.urlopen(url, timeout=5) as response:
        if response.status != 200:
            raise RuntimeError("Model server was not healthy")
    precache_model_thread.join(timeout=max(0, SERVER_STARTUP_TIMEOUT-(time.time()-NOTEBOOK_START_TIME)))
    if precache_model_thread.is_alive():
        raise TimeoutError("Background weight prefetch still active; benchmark cannot start")
    _write_state("ready", background_prefetch_joined=True)
    code = proc.wait()
    if code:
        raise RuntimeError("Model server exited: " + str(code))
finally:
    stopped = True if proc is None else stop_group(proc, grace=10)
    _write_state("stopped", server_stopped=stopped)


In [ ]:
%%writefile /kaggle/working/019-code/src/arc3_research/__init__.py
"""Small research interventions and manual experiment runners for ARC-AGI-3."""


In [ ]:
%%writefile /kaggle/working/019-code/src/arc3_research/experiment_config.py
"""Narrow, explicit research-arm environment overrides (never credentials)."""
from __future__ import annotations

import json
from typing import Any


ALLOWED_ARM_ENVIRONMENT = {
    "ARC3_PRIORITY_TAIL_LOOKUP": frozenset({"remaining", "1"}),
    "ARC3_PRIORITY_PACE": frozenset({"0", "1"}),
}


def arm_environment(value: Any) -> dict[str, str]:
    if not isinstance(value, dict):
        raise ValueError("Arm environment must be a JSON object.")
    result = {}
    for key, setting in value.items():
        if key not in ALLOWED_ARM_ENVIRONMENT or not isinstance(setting, str):
            raise ValueError(f"Unsupported research environment key/type: {key!r}")
        if setting not in ALLOWED_ARM_ENVIRONMENT[key]:
            raise ValueError(f"Unsupported setting for {key}: {setting!r}")
        result[key] = setting
    return result


def parse_arm_environment(value: str) -> dict[str, str]:
    return arm_environment(json.loads(value))


def validate_config(config: dict[str, Any]) -> None:
    arms = config["arm_order"]
    if not arms or len(arms) != len(set(arms)):
        raise ValueError("arm_order must contain distinct arms.")
    environments = config.get("arm_environment", {})
    if not isinstance(environments, dict) or set(environments) - set(arms):
        raise ValueError("arm_environment must name only configured arms.")
    for value in environments.values():
        arm_environment(value)
    if "tail_80k" in arms:
        expected = {
            "baseline": {"ARC3_PRIORITY_TAIL_LOOKUP": "remaining", "ARC3_PRIORITY_PACE": "0"},
            "tail_80k": {"ARC3_PRIORITY_TAIL_LOOKUP": "1", "ARC3_PRIORITY_PACE": "0"},
        }
        if set(arms) != set(expected) or environments != expected:
            raise ValueError("tail_80k requires the frozen baseline/tail-only environment pair.")
        if config.get("active_streams") != 10 or config.get("n_passes_per_arm") != 1:
            raise ValueError("The tail comparison must retain 10 streams and one pass.")
        if not all(config.get(key) is True for key in (
            "inference_timing", "scheduler_observer", "request_deadline",
            "strict_arm_budget", "require_cache_flush", "preflight_before_serving",
        )):
            raise ValueError("The tail comparison requires paired diagnostics, deadline checks and strict startup contracts.")
    if "pace_1x" in arms:
        expected = {
            "baseline": {"ARC3_PRIORITY_TAIL_LOOKUP": "remaining", "ARC3_PRIORITY_PACE": "0"},
            "pace_1x": {"ARC3_PRIORITY_TAIL_LOOKUP": "remaining", "ARC3_PRIORITY_PACE": "1"},
        }
        if set(arms) != set(expected) or environments != expected:
            raise ValueError("pace_1x requires the frozen baseline/pace-only environment pair.")
        if config.get("active_streams") != 10 or config.get("n_passes_per_arm") != 1:
            raise ValueError("The pace comparison must retain 10 streams and one pass.")
        if not all(config.get(key) is True for key in (
            "inference_timing", "scheduler_observer", "request_deadline",
            "strict_arm_budget", "require_cache_flush", "preflight_before_serving",
        )):
            raise ValueError("The pace comparison requires paired diagnostics, deadline checks and strict startup contracts.")
    if any(a in ("native_values", "native") for a in arms):
        # Scheduler frozen to baseline; single-factor is native handoff only.
        for arm in arms:
            if environments.get(arm, {}) != {"ARC3_PRIORITY_TAIL_LOOKUP": "remaining", "ARC3_PRIORITY_PACE": "0"}:
                raise ValueError("native_values arms require frozen baseline scheduler.")
        if config.get("active_streams") != 10 or config.get("n_passes_per_arm") != 1:
            raise ValueError("Native comparison must retain 10 streams and one pass.")
    if "observation_grounding" in arms:
        expected = {arm: {"ARC3_PRIORITY_TAIL_LOOKUP": "remaining", "ARC3_PRIORITY_PACE": "0"}
                    for arm in ("baseline", "observation_grounding")}
        if arms != ["baseline", "observation_grounding"] or environments != expected:
            raise ValueError("018 requires baseline then observation_grounding with unchanged scheduling.")
        if (config.get("active_streams") != 10 or config.get("n_passes_per_arm") != 1
                or config.get("warmup_action_games") != 0
                or config.get("reference_arm") != "baseline"):
            raise ValueError("018 requires 10 stream capacity, one pass, no warmup and baseline reference.")
        if not all(config.get(key) is True for key in (
            "inference_timing", "scheduler_observer", "request_deadline",
            "strict_arm_budget", "require_cache_flush", "require_cache_flush_log",
            "require_drained_workers", "require_complete_pool", "preflight_before_serving",
            "native_payload_cells",
        )):
            raise ValueError("018 requires the shared strict budget, diagnostics and startup contracts.")


In [ ]:
%%writefile /kaggle/working/019-code/src/arc3_research/session_runner.py
"""Run configured isolated CPU harnesses against one loaded GPU server."""
from __future__ import annotations

import json
import os
from pathlib import Path
import signal
import subprocess
import sys
import threading
import time
from urllib.request import Request, urlopen

from arc3_research.experiment_config import validate_config


def stop_group(process: subprocess.Popen, grace: float = 10.0) -> bool:
    """Stop an owned start_new_session group, even after its leader exits."""
    if getattr(process, "_stopped_group_verified", False):
        return True

    def signal_group(sig: int) -> bool:
        # Reap our leader before probing. On macOS a zombie leader can make a
        # vanished group report EPERM until poll() reaps it. Retry that race once;
        # a genuine permission error remains an error, never a success claim.
        for attempt in range(2):
            process.poll()
            try:
                os.killpg(process.pid, sig)
                return True
            except ProcessLookupError:
                return False
            except PermissionError:
                if attempt:
                    raise
        return False

    def group_exists() -> bool:
        return signal_group(0)

    if not signal_group(signal.SIGTERM):
        process._stopped_group_verified = True
        return True
    until = time.monotonic() + max(0.0, grace)
    while group_exists() and time.monotonic() < until:
        process.poll()  # reap our leader without assuming its children exited
        time.sleep(0.05)
    if group_exists():
        signal_group(signal.SIGKILL)
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        return False
    until = time.monotonic() + 2.0
    while group_exists() and time.monotonic() < until:
        time.sleep(0.05)
    stopped = not group_exists()
    if stopped:
        # Never send a signal to this old PGID again after an hours-long arm.
        process._stopped_group_verified = True
    return stopped


def gpu_sample(path: Path, phase: str) -> dict:
    started = time.time()
    sample = {"time": started, "phase": phase}
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total,power.draw",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=5,
        )
        sample.update(metrics=result.stdout.strip(), returncode=result.returncode)
        if result.returncode:
            sample["error"] = result.stderr.strip()[:500]
    except (OSError, subprocess.TimeoutExpired) as exc:
        sample.update(metrics="", returncode=-1, error=str(exc))
    sample["observed_at_epoch"] = time.time()
    sample["sampling_seconds"] = sample["observed_at_epoch"] - started
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a") as file:
        file.write(json.dumps(sample) + "\n")
    return sample


class GPUSampler:
    """Append every observation through startup, solving and shutdown."""

    def __init__(self, path: Path, interval_seconds: float = 10.0) -> None:
        self.path = Path(path)
        self.interval_seconds = max(1.0, float(interval_seconds))
        self.phase = "startup"
        self.samples = 0
        self.errors = 0
        self._lock = threading.Lock()
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._sample_loop,
                                        name="research-gpu-sampler", daemon=True)
        self._thread.start()

    def sample_now(self) -> None:
        with self._lock:
            sample = gpu_sample(self.path, self.phase)
            self.samples += 1
            self.errors += sample["returncode"] != 0

    def set_phase(self, phase: str) -> None:
        with self._lock:
            self.phase = phase
        self.sample_now()

    def _sample_loop(self) -> None:
        while not self._stop.is_set():
            started = time.monotonic()
            self.sample_now()
            self._stop.wait(max(0.0, self.interval_seconds - (time.monotonic() - started)))

    def stop(self) -> dict:
        self._stop.set()
        self._thread.join(timeout=6)
        self.sample_now()
        return {"path": str(self.path), "interval_seconds": self.interval_seconds,
                "sample_count": self.samples, "error_count": self.errors,
                "stopped": not self._thread.is_alive()}


def start_gpu_sampling(path: Path, interval_seconds: float = 10.0) -> GPUSampler:
    return GPUSampler(path, interval_seconds)


def run_session(namespace: dict) -> dict:
    root = Path(namespace["WORKING_DIR"])
    config = namespace["ARC3_EXPERIMENT"]
    server = namespace["proc"]
    start = namespace["ARC3_SESSION_START"]
    server_url = "http://127.0.0.1:" + str(namespace["SERVED_MODEL_PORT"])
    arms = config["arm_order"]
    monitor = namespace.get("ARC3_GPU_MONITOR")
    if monitor is None:
        monitor = start_gpu_sampling(root / "gpu-utilization.jsonl",
                                     config.get("gpu_sample_interval_seconds", 10))
    report = {"experiment": config, "started_at_epoch": start,
              "arms": [], "source_hashes": namespace["ARC3_SOURCE_HASHES"]}
    report_path = root / "experiment-manifest.json"

    def save():
        report["elapsed_seconds"] = time.time() - start
        report_path.write_text(json.dumps(report, indent=2))

    save()
    try:
        validate_config(config)
        if not arms or len(set(arms)) != len(arms):
            raise ValueError("arm_order must be nonempty with unique output names")
        monitor.set_phase("waiting_for_server")
        # The inherited launcher may return before the model is ready. Do not
        # spend the full research allocation on failed loading/retrying.
        ready = False
        while time.time() < start + config["startup_limit_seconds"]:
            if server.poll() is not None:
                raise RuntimeError("Model server exited during startup; inspect serve.log")
            try:
                with urlopen(server_url + "/health", timeout=5) as response:
                    ready = response.status == 200
                if ready:
                    break
            except Exception:
                pass
            time.sleep(5)
        if not ready:
            raise RuntimeError("Model was not ready within the startup allocation")

        report["model_ready_seconds"] = time.time() - start
        remaining = config["hard_limit_seconds"] - (time.time() - start)
        # Each CPU runner has at most 120s drain + 30s output allowance.
        # Reserve another minute for outer shutdown and the notebook watchdog.
        cleanup_reserve = 150 * len(arms) + 60
        arm_budget = min(config["requested_arm_seconds"],
                         int((remaining - cleanup_reserve) / len(arms)))
        if config.get("strict_arm_budget") and arm_budget != config["requested_arm_seconds"]:
            raise RuntimeError("Startup consumed the frozen solver budget; aborting instead of shortening the comparison.")
        if arm_budget < 600:
            raise RuntimeError("Insufficient budget left for the configured experiment")
        report["equal_arm_budget_seconds"] = arm_budget
        report["cleanup_reserve_seconds"] = cleanup_reserve
        report["arm_count"] = len(arms)
        save()
        for arm in arms:
            out = root / arm
            out.mkdir(parents=True, exist_ok=True)
            entry = {"arm": arm, "budget_seconds": arm_budget,
                     "started_at_epoch": time.time()}
            report["arms"].append(entry)
            monitor.set_phase(arm + "_cache_flush")
            serve_log = root / "serve.log"
            flush_offset = serve_log.stat().st_size if serve_log.is_file() else None
            # Flush prompt caches only, retaining weights and compiled kernels.
            # If unsupported, report the limitation rather than reload the GPU.
            try:
                request = Request(server_url + "/flush_cache", data=b"", method="POST")
                with urlopen(request, timeout=30) as response:
                    entry["cache_flush_http_status"] = response.status
                    entry["cache_flush_body"] = response.read(512).decode(errors="replace")
            except Exception as exc:
                entry["cache_flush_error"] = str(exc)
            save()
            if config.get("require_cache_flush") and entry.get("cache_flush_http_status") != 200:
                raise RuntimeError(f"Cache flush failed before {arm}; refusing unmatched cache state.")
            if config.get("require_cache_flush_log"):
                # This SGLang version returns HTTP200 even when active requests
                # prevent flushing. Require its new backend success log too.
                confirmed = False
                if flush_offset is not None:
                    until = time.monotonic() + 30.0
                    while time.monotonic() < until:
                        with serve_log.open("rb") as stream:
                            stream.seek(flush_offset)
                            recent = stream.read(131072)
                        if b"Cache flushed successfully!" in recent:
                            confirmed = True
                            break
                        if server.poll() is not None:
                            break
                        time.sleep(0.2)
                entry["cache_flush_backend_confirmed"] = confirmed
                save()
                if not confirmed:
                    raise RuntimeError(f"Backend did not confirm cache flush before {arm}; aborting comparison.")

            command = [sys.executable, "-u", "-m", "arc3_research.run_arm",
                       "--arm", arm, "--bundle-dir", str(namespace["BUNDLE_DIR"]),
                       "--working-dir", str(out), "--game-list", ",".join(config["games"]),
                       "--budget-seconds", str(arm_budget)]
            if config.get("inference_timing", False):
                command.append("--inference-timing")
            if config.get("request_deadline", False):
                command.append("--request-deadline")
            if config.get("scheduler_observer", False):
                command.append("--scheduler-observer")
            command.extend(["--environment-json", json.dumps(config.get("arm_environment", {}).get(arm, {}))])
            env = os.environ.copy()
            env["PYTHONPATH"] = "/kaggle/research/src" + os.pathsep + env.get("PYTHONPATH", "")
            print(f"Starting {arm}: {len(config['games'])} games, {arm_budget}s, shared GPU", flush=True)
            monitor.set_phase(arm)
            with (out / "run.log").open("w") as log:
                child = subprocess.Popen(command, env=env, stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)
                namespace["ARC3_CHILD_PROCESSES"].append(child)
                try:
                    next_status = 0.0
                    while child.poll() is None:
                        now = time.time()
                        if server.poll() is not None:
                            entry["model_server_exited_during_arm"] = server.returncode
                            stop_group(child)
                            break
                        if now >= next_status:
                            print(f"{arm}: elapsed {now-entry['started_at_epoch']:.0f}s; "
                                  f"session {now-start:.0f}s", flush=True)
                            next_status = now + 60
                        if now - entry["started_at_epoch"] > arm_budget + 150:
                            entry["parent_timeout"] = True
                            stop_group(child)
                            break
                        time.sleep(3)
                finally:
                    monitor.set_phase(arm + "_cleanup")
                    stop_group(child)
                entry["exit_code"] = child.returncode
            entry["elapsed_seconds"] = time.time() - entry["started_at_epoch"]
            entry["benchmark_exists"] = (out / "benchmark.json").exists()
            arm_meta_path = out / "arm-metadata.json"
            if arm_meta_path.exists():
                arm_meta = json.loads(arm_meta_path.read_text())
                entry["game_state_counts"] = arm_meta.get("game_state_counts", {})
                results = arm_meta.get("game_results", [])
                entry["all_games_failed_to_run"] = not results or all(
                    game.get("state") in {"crashed", "not_started"} for game in results
                )
                entry["lingering_worker_threads"] = arm_meta.get("lingering_worker_threads", [])
                entry["environment_overrides"] = arm_meta.get("environment_overrides", {})
                entry["complete_pool"] = (
                    len(results) == len(config["games"])
                    and {game.get("game_id") for game in results} == set(config["games"])
                    # Terminal states are won/cancelled/gave_up; crashes/not_started fail.
                    # gave_up carries valid score/levels (016 baseline: 8 gave_up with
                    # scores, not crashes) and must not waste a full session.
                    and all(game.get("state") in {"won", "cancelled", "gave_up"} for game in results)
                    and not any(game.get("state") in {"crashed", "not_started"} for game in results)
                )
            tail = (out / "run.log").read_text(errors="replace")[-5000:]
            print(tail, flush=True)
            save()
            if entry["exit_code"] != 0:
                # A broken baseline gives no useful paired comparison. Stop
                # the session instead of spending another arm on that failure.
                raise RuntimeError(f"Arm {arm} exited {entry['exit_code']}; see {arm}/run.log")
            if entry.get("all_games_failed_to_run") or not entry["benchmark_exists"]:
                raise RuntimeError(f"Arm {arm} produced no usable game runs; see {arm}/run.log")
            if config.get("require_drained_workers") and entry.get("lingering_worker_threads"):
                raise RuntimeError(f"Arm {arm} has undrained workers; stopping before another arm.")
            if config.get("require_complete_pool") and not entry.get("complete_pool"):
                raise RuntimeError(f"Arm {arm} has incomplete/crashed game coverage; stopping before another arm.")
        report["status"] = "completed"
    except BaseException as exc:
        report["status"] = "failed"
        report["error"] = repr(exc)
        raise
    finally:
        monitor.set_phase("shutdown")
        cleanup_errors = []
        for child in namespace["ARC3_CHILD_PROCESSES"]:
            try:
                if not stop_group(child):
                    cleanup_errors.append(f"CPU process group {child.pid} still exists")
            except Exception as exc:
                cleanup_errors.append(f"CPU group cleanup: {type(exc).__name__}: {exc}")
        try:
            report["gpu_server_process_group_stopped"] = stop_group(server)
        except Exception as exc:
            report["gpu_server_process_group_stopped"] = False
            cleanup_errors.append(f"GPU group cleanup: {type(exc).__name__}: {exc}")
        report["gpu_server_leader_exited"] = server.poll() is not None
        report["gpu_server_stopped"] = report["gpu_server_process_group_stopped"]
        if cleanup_errors or not report["gpu_server_stopped"]:
            report["status"] = "failed"
            report["cleanup_errors"] = cleanup_errors
        monitor.set_phase("finished")
        report["gpu_sampling"] = monitor.stop()
        save()
    if report["status"] != "completed":
        raise RuntimeError("Session cleanup did not confirm all owned process groups stopped.")
    return report


In [ ]:
%%writefile /kaggle/working/019-code/src/arc3_research/ple_weights.py
"""Bounded, CPU-only checkpoint audit for experiment 019.

Reads JSON, safetensors headers and selected byte ranges. Never imports torch,
unpickles data, loads a complete weight tensor, or contacts a network service.
The result is a sampling contract, not a full checkpoint integrity proof.
"""
from __future__ import annotations

import hashlib
import json
import math
import os
from pathlib import Path
import re
import struct
import time


PLE_PREFIX = "model.language_model.layers.1.ple.ple_embedding.ngram_embedding."
SCALE_KEY = PLE_PREFIX + "weight_scale"
PLE_RE = re.compile(re.escape(PLE_PREFIX) + r"shard_(\d+)\.weight$")
DTYPE_BITS = {"BOOL": 8, "U8": 8, "I8": 8, "F8_E4M3": 8,
              "F8_E5M2": 8, "I16": 16, "U16": 16, "F16": 16,
              "BF16": 16, "I32": 32, "U32": 32, "F32": 32,
              "I64": 64, "U64": 64, "F64": 64}


class AuditStop(RuntimeError):
    pass


def _require(condition, message):
    if not condition:
        raise AuditStop(message)


def _sha(data):
    return hashlib.sha256(data).hexdigest()


def _unique(pairs):
    result = {}
    for key, value in pairs:
        _require(key not in result, f"Duplicate JSON key: {key}")
        result[key] = value
    return result


def main():
    import argparse
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--bf16-dir", required=True)
    parser.add_argument("--fp8-dir", required=True)
    parser.add_argument("--draft-dir", required=True)
    parser.add_argument("--contract", required=True)
    parser.add_argument("--output", required=True)
    args = parser.parse_args()
    report = run_weight_audit(args.bf16_dir, args.fp8_dir, args.draft_dir,
                              args.contract, args.output)
    print(json.dumps({"passed": report["passed"], "stage": report["stage"],
                      "elapsed_seconds": report["elapsed_seconds"],
                      "error": report.get("error"), "output": args.output}))
    return 0 if report["passed"] else 1


def _json(raw):
    return json.loads(raw, object_pairs_hook=_unique)


class _Reader:
    def __init__(self, root, deadline, budget):
        self.root = Path(root).resolve()
        self.deadline = deadline
        self.budget = int(budget)
        self.bytes_read = 0
        self.read_calls = 0
        self.fds = {}

    def check(self):
        _require(time.monotonic() < self.deadline, "Audit wall-clock deadline exceeded")

    def path(self, name):
        rel = Path(name)
        _require(not rel.is_absolute() and ".." not in rel.parts,
                 f"Unsafe input-relative path: {name}")
        path = (self.root / rel).resolve()
        _require(path.is_relative_to(self.root), f"Input path escapes root: {name}")
        _require(path.is_file(), f"Missing input: {path}")
        return path

    def read(self, name, start, count):
        self.check()
        _require(start >= 0 and count >= 0, "Negative range")
        _require(self.bytes_read + count <= self.budget,
                 f"Read budget exceeded for {self.root}")
        path = self.path(name)
        _require(start + count <= path.stat().st_size, f"Range beyond file: {name}")
        if name not in self.fds:
            self.fds[name] = os.open(path, os.O_RDONLY)
        data = os.pread(self.fds[name], count, start)
        self.bytes_read += len(data)
        self.read_calls += 1
        _require(len(data) == count, f"Short read: {name}")
        self.check()
        return data

    def document(self, name, limit=32 * 1024 * 1024):
        size = self.path(name).stat().st_size
        _require(size <= limit, f"Metadata file too large: {name}")
        raw = self.read(name, 0, size)
        return _json(raw), _sha(raw)

    def close(self):
        for fd in self.fds.values():
            os.close(fd)


def _catalog(reader, max_header):
    index, index_sha = reader.document("model.safetensors.index.json")
    wm = index.get("weight_map")
    _require(isinstance(wm, dict) and wm, "Missing/empty weight_map")
    _require(all(isinstance(k, str) and isinstance(v, str) for k, v in wm.items()),
             "Invalid index keys/files")
    descriptors, files = {}, {}
    for name in sorted(set(wm.values())):
        reader.check()
        n = struct.unpack("<Q", reader.read(name, 0, 8))[0]
        _require(0 < n <= max_header, f"Invalid header length: {name}: {n}")
        raw = reader.read(name, 8, n)
        header = _json(raw)
        size = reader.path(name).stat().st_size
        spans = []
        count = 0
        for key, d in header.items():
            if key == "__metadata__":
                continue
            _require(isinstance(d, dict), f"Invalid descriptor: {key}")
            dtype, shape, offsets = d.get("dtype"), d.get("shape"), d.get("data_offsets")
            _require(dtype in DTYPE_BITS, f"Unsupported dtype {dtype}: {key}")
            _require(isinstance(shape, list) and all(type(v) is int and v >= 0 for v in shape),
                     f"Invalid shape: {key}")
            _require(isinstance(offsets, list) and len(offsets) == 2
                     and all(type(v) is int for v in offsets), f"Invalid offsets: {key}")
            start, end = offsets
            expected = math.prod(shape) * DTYPE_BITS[dtype] // 8
            _require(0 <= start <= end and end - start == expected
                     and 8 + n + end <= size, f"Bad tensor bounds/size: {key}")
            _require(key not in descriptors, f"Tensor occurs in multiple files: {key}")
            _require(wm.get(key) == name, f"Header/index mapping mismatch: {key}")
            descriptors[key] = {"file": name, "dtype": dtype, "shape": shape,
                                "offset": 8 + n + start, "nbytes": expected}
            spans.append((start, end))
            count += 1
        cursor = 0
        for start, end in sorted(spans):
            _require(start == cursor, f"Overlapping/non-contiguous tensor data: {name}")
            cursor = end
        _require(8 + n + cursor == size, f"Trailing/unindexed tensor bytes: {name}")
        files[name] = {"file_size": size, "header_size": n,
                       "header_sha256": _sha(raw), "tensor_count": count}
    _require(set(descriptors) == set(wm), "Index refers to missing tensor keys")
    return {"index_sha256": index_sha, "index_tensor_count": len(wm),
            "files": files, "tensors": descriptors}


def decode_e4m3fn(byte):
    """IEEE-like E4M3FN: exponent 15 is finite except mantissa 7 (NaN)."""
    sign = -1.0 if byte & 128 else 1.0
    exponent, mantissa = (byte >> 3) & 15, byte & 7
    if exponent == 15 and mantissa == 7:
        return math.nan
    if exponent == 0:
        return sign * math.ldexp(mantissa, -9)
    return sign * math.ldexp(1.0 + mantissa / 8.0, exponent - 7)


def decode_bf16(raw):
    _require(len(raw) % 2 == 0, "Odd BF16 byte count")
    return [struct.unpack("<f", struct.pack("<I", v[0] << 16))[0]
            for v in struct.iter_unpack("<H", raw)]


def _round_bf16(value):
    bits = struct.unpack("<I", struct.pack("<f", value))[0]
    if bits & 0x7f800000 != 0x7f800000:
        bits = (bits + 0x7fff + ((bits >> 16) & 1)) & 0xffffffff
    return struct.unpack("<f", struct.pack("<I", bits & 0xffff0000))[0]


def _sample_rows(key, rows, random_count, seed):
    chosen = {0, rows - 1, rows // 2}
    index = 0
    count = min(rows, 3 + random_count)
    while len(chosen) < count:
        digest = hashlib.sha256(f"{seed}|{key}|{index}".encode()).digest()
        chosen.add(int.from_bytes(digest[:8], "little") % rows)
        index += 1
    return sorted(chosen)


def _sample_keys(keys, tensors, maximum, seed):
    """Fixed strata: key roles, each physical shard, model layers, then hash fill."""
    priority = lambda k: (_sha((seed + "|" + k).encode()), k)
    selected = []

    def add(key):
        if key not in selected and len(selected) < maximum:
            selected.append(key)

    special = [k for k in keys if any(s in k for s in
               ("embed_tokens.weight", "lm_head.weight", ".ple.", "hyper_connection_mixer"))]
    for key in sorted(special, key=priority)[:24]:
        add(key)
    groups = {}
    for key in keys:
        groups.setdefault(tensors[key]["file"], []).append(key)
    for filename in sorted(groups):
        for key in sorted(groups[filename], key=priority)[:2]:
            add(key)
    layers = {}
    for key in keys:
        match = re.search(r"\.layers\.(\d+)\.", key)
        if match:
            layers.setdefault(int(match[1]), []).append(key)
    for layer in sorted(layers):
        add(min(layers[layer], key=priority))
    for key in sorted(keys, key=priority):
        if len(selected) >= maximum:
            break
        add(key)
    return selected


def _tensor_sample(reader, desc, span_bytes):
    size = desc["nbytes"]
    width = min(span_bytes, size)
    starts = sorted({0, max(0, (size - width) // 2), max(0, size - width)})
    h = hashlib.sha256()
    spans = []
    for start in starts:
        raw = reader.read(desc["file"], desc["offset"] + start, width)
        h.update(struct.pack("<QQ", start, width))
        h.update(raw)
        spans.append([start, width])
    return {"sha256": h.hexdigest(), "spans": spans}


def run_weight_audit(bf16_dir, fp8_dir, draft_dir, contract, out):
    """Return and write a fail-closed report; callers must require passed=True.

    `contract` is a dict or path to 019-weight-contract.json. `out` is a JSON
    filename. The elapsed-time deadline is checked before/after every bounded
    read; a parent process timeout is still needed for a hung filesystem read.
    """
    started = time.monotonic()
    cfg = _json(Path(contract).read_bytes()) if isinstance(contract, (str, Path)) else contract
    limits = cfg["audit_limits"]
    deadline = started + limits["max_seconds"]
    readers = {name: _Reader(root, deadline, limits["max_bytes_per_source"])
               for name, root in [("bf16", bf16_dir), ("fp8", fp8_dir), ("draft", draft_dir)]}
    result = {"schema_version": 1, "passed": False, "stage": "metadata",
              "contract_sha256": _sha(json.dumps(cfg, sort_keys=True, separators=(",", ":")).encode()),
              "limits": limits, "sources": {}, "checks": {},
              "limitations": ["Selected payload ranges do not prove full tensor/file identity.",
                              "No model loading, inference, GPU execution or game data was used.",
                              "PLE error limits are engineering smoke gates, not quality guarantees.",
                              "Audit touches file cache; later startup is not an untouched cold-I/O measurement."]}
    try:
        configs = {}
        cats = {}
        for arm, reader in readers.items():
            config, sha = reader.document("config.json")
            configs[arm] = config
            result["sources"][arm] = {"path": str(reader.root), "config_sha256": sha}
        a = json.loads(json.dumps(configs["bf16"]))
        b = json.loads(json.dumps(configs["fp8"]))
        _require(b["text_config"].pop("ple_embedding_dtype", None) == "float8_e4m3fn",
                 "Candidate must explicitly declare float8_e4m3fn PLE")
        _require(a["text_config"].get("ple_embedding_dtype") in (None, "bfloat16"),
                 "Baseline PLE dtype is not BF16")
        a["text_config"].pop("ple_embedding_dtype", None)
        _require(a == b, "Target configs differ beyond PLE storage dtype")
        config_sha = _sha(json.dumps(a, sort_keys=True, separators=(",", ":")).encode())
        _require(config_sha == cfg["expected_normalized_config_sha256"],
                 "Target config differs from pinned upstream contract")
        for key, expected in cfg["expected_text_config"].items():
            _require(a["text_config"].get(key) == expected, f"Unexpected text_config.{key}")
        q = a.get("quantization_config", {})
        _require(q.get("quant_method") == "auto-round" and q.get("bits") == 4
                 and q.get("group_size") == 128 and q.get("sym") is True,
                 "Target must remain AutoRound symmetric W4A16/group128")
        result["checks"]["config_only_ple_dtype_diff"] = True
        for name in cfg["identical_target_metadata"]:
            hashes = {}
            for arm in ("bf16", "fp8"):
                r = readers[arm]
                size = r.path(name).stat().st_size
                _require(size <= limits["max_metadata_file_bytes"], f"Metadata too large: {name}")
                hashes[arm] = _sha(r.read(name, 0, size))
            _require(hashes["bf16"] == hashes["fp8"], f"Target metadata differs: {name}")
            if name == "tokenizer.json" and cfg.get("expected_tokenizer_sha256"):
                _require(hashes["bf16"] == cfg["expected_tokenizer_sha256"], "Wrong tokenizer for frozen FR-Spec map")
            result["checks"][name] = hashes["bf16"]
        dq = configs["draft"].get("quantization_config", {})
        _require(dq.get("quant_method") == "compressed-tensors"
                 and "mtp_routed_experts" in dq.get("config_groups", {}),
                 "Independent draft is not expected compressed-tensors MTP")
        group = dq["config_groups"]["mtp_routed_experts"]
        weights = group.get("weights", {})
        _require(weights.get("num_bits") == 4 and weights.get("group_size") == 32
                 and weights.get("symmetric") is True,
                 "Draft must remain symmetric INT4 group32")
        for arm, reader in readers.items():
            cats[arm] = _catalog(reader, limits["max_header_bytes"])
            result["sources"][arm].update({k: v for k, v in cats[arm].items() if k != "tensors"})
        tensors = {arm: cat["tensors"] for arm, cat in cats.items()}
        _require(any(k.startswith("mtp.layers.0.mlp.experts.") for k in tensors["draft"]),
                 "Independent draft has no expected MTP expert keys")
        result["stage"] = "tensor_contract"
        ple = cfg["ple"]
        expected_keys = [PLE_PREFIX + f"shard_{i}.weight" for i in range(ple["shard_count"])]
        for arm in ("bf16", "fp8"):
            actual = {k for k in tensors[arm] if PLE_RE.fullmatch(k)}
            _require(actual == set(expected_keys), f"Missing/extra PLE shards: {arm}")
            wanted_prefix = set(expected_keys) | ({SCALE_KEY} if arm == "fp8" else set())
            _require({k for k in tensors[arm] if k.startswith(PLE_PREFIX)} == wanted_prefix,
                     f"Unexpected PLE scale/storage keys: {arm}")
            for key in expected_keys:
                d = tensors[arm][key]
                _require(d["shape"] == ple["shard_shape"]
                         and d["dtype"] == ple[f"{arm}_dtype"], f"PLE shape/dtype mismatch: {arm}: {key}")
        nonple = {arm: {k for k in tensors[arm] if not k.startswith(PLE_PREFIX)
                       and not k.startswith("mtp.")} for arm in ("bf16", "fp8")}
        _require(nonple["bf16"] == nonple["fp8"], "Non-PLE/non-MTP key sets differ")
        keys_sha = _sha(json.dumps(sorted(nonple["bf16"]), separators=(",", ":")).encode())
        _require(len(nonple["bf16"]) == cfg["expected_nonple_tensor_count"]
                 and keys_sha == cfg["expected_nonple_keys_sha256"],
                 "Non-PLE key set differs from pinned upstream contract")
        for key in nonple["bf16"]:
            da, db = tensors["bf16"][key], tensors["fp8"][key]
            _require((da["dtype"], da["shape"], da["nbytes"]) ==
                     (db["dtype"], db["shape"], db["nbytes"]), f"Non-PLE header mismatch: {key}")
        result["checks"]["nonple_all_headers_equal"] = len(nonple["bf16"])
        result["checks"]["nonple_keys_sha256"] = keys_sha
        result["checks"]["excluded_mtp_counts"] = {
            arm: sum(k.startswith("mtp.") for k in tensors[arm]) for arm in ("bf16", "fp8")}
        scale_d = tensors["fp8"].get(SCALE_KEY)
        _require(scale_d and scale_d["shape"] == [1] and scale_d["dtype"] == ple["scale_dtype"],
                 "Missing/invalid global scale")
        raw = readers["fp8"].read(scale_d["file"], scale_d["offset"], scale_d["nbytes"])
        scale = decode_bf16(raw)[0] if scale_d["dtype"] == "BF16" else struct.unpack("<f", raw)[0]
        _require(math.isfinite(scale) and scale > 0, "Scale must be positive finite")
        _require(raw.hex() == ple["scale_raw_hex"], "Scale differs from pinned upstream value")
        result["scale"] = {"value": scale, "dtype": scale_d["dtype"], "shape": [1],
                           "raw_hex": raw.hex(), "formula": "BF16(FP8_value * BF16_scale)"}
        result["stage"] = "nonple_samples"
        sample = cfg["sampling"]
        selected = _sample_keys(nonple["bf16"], tensors["bf16"], sample["nonple_max_tensors"], sample["seed"])
        result["nonple_samples"] = []
        for key in selected:
            row = {"key": key}
            for arm in ("bf16", "fp8"):
                row[arm] = _tensor_sample(readers[arm], tensors[arm][key], sample["tensor_span_bytes"])
            result["nonple_samples"].append(row)
            _require(row["bf16"] == row["fp8"], f"Sampled non-PLE payload differs: {key}")
        result["draft_samples"] = []
        for key in _sample_keys(set(tensors["draft"]), tensors["draft"], sample["draft_max_tensors"], sample["seed"]):
            result["draft_samples"].append({"key": key, **_tensor_sample(
                readers["draft"], tensors["draft"][key], sample["tensor_span_bytes"])})
        result["stage"] = "ple_samples"
        result["ple_samples"] = []
        total_n = 0
        sumsq_ref = sumsq_err = sumsq_unscaled = sumsq_inverse = max_abs_error = 0.0
        max_bound_excess = 0.0
        violations = nonfinite = saturated = zeros = 0
        for key in expected_keys:
            rows, width = ple["shard_shape"]
            chosen = _sample_rows(key, rows, sample["random_rows_per_shard"], sample["seed"])
            ha, hb = hashlib.sha256(), hashlib.sha256()
            shard_sse = shard_ref = 0.0
            for row in chosen:
                da, db = tensors["bf16"][key], tensors["fp8"][key]
                ba = readers["bf16"].read(da["file"], da["offset"] + row * width * 2, width * 2)
                bb = readers["fp8"].read(db["file"], db["offset"] + row * width, width)
                ha.update(struct.pack("<Q", row) + ba)
                hb.update(struct.pack("<Q", row) + bb)
                for ref, code in zip(decode_bf16(ba), bb):
                    qvalue = decode_e4m3fn(code)
                    if not math.isfinite(ref) or not math.isfinite(qvalue):
                        nonfinite += 1
                        continue
                    reconstructed = _round_bf16(qvalue * scale)
                    error = abs(reconstructed - ref)
                    bound = sample["relative_error_bound"] * abs(ref) + scale * sample["absolute_error_scale_factor"]
                    violations += error > bound
                    max_bound_excess = max(max_bound_excess, error - bound)
                    saturated += abs(qvalue) == 448.0
                    zeros += qvalue == 0.0
                    max_abs_error = max(max_abs_error, error)
                    total_n += 1
                    sumsq_ref += ref * ref
                    sumsq_err += error * error
                    sumsq_unscaled += (qvalue - ref) ** 2
                    sumsq_inverse += (qvalue / scale - ref) ** 2
                    shard_sse += error * error
                    shard_ref += ref * ref
            result["ple_samples"].append({"key": key, "rows": chosen,
                "bf16_sample_sha256": ha.hexdigest(), "fp8_sample_sha256": hb.hexdigest(),
                "nrmse": math.sqrt(shard_sse / shard_ref) if shard_ref else None})
        nrmse = math.sqrt(sumsq_err / sumsq_ref) if sumsq_ref else None
        result["ple_numeric"] = {"elements": total_n, "nonfinite_elements": nonfinite,
            "scale_times_q_nrmse": nrmse, "max_abs_error": max_abs_error,
            "error_bound_violations": violations, "max_error_bound_excess": max_bound_excess,
            "saturated_fp8_elements": saturated, "zero_fp8_elements": zeros,
            "unscaled_nrmse": math.sqrt(sumsq_unscaled / sumsq_ref) if sumsq_ref else None,
            "inverse_scale_nrmse": math.sqrt(sumsq_inverse / sumsq_ref) if sumsq_ref else None}
        _require(nonfinite == 0, "Nonfinite PLE sample")
        _require(nrmse is not None and nrmse <= sample["max_nrmse"], "PLE sample NRMSE gate failed")
        _require(violations == 0, "PLE sample error exceeds frozen conservative FP8 bound")
        result["stage"] = "complete"
        result["passed"] = True
    except Exception as exc:
        result["error"] = {"type": type(exc).__name__, "message": str(exc)}
    finally:
        result["elapsed_seconds"] = time.monotonic() - started
        result["io"] = {arm: {"bytes_read": r.bytes_read, "read_calls": r.read_calls}
                        for arm, r in readers.items()}
        for reader in readers.values():
            reader.close()
        destination = Path(out)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text(json.dumps(result, ensure_ascii=False, indent=2, allow_nan=False) + "\n")
    return result


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/019-code/src/arc3_research/ple_microbench.py
"""019 fixed-request PLE microbenchmark; imports and preparation are offline.

Only run_phase() / CLI ``run --allow-local-network`` contact an existing loopback
service. No service startup, model code execution, game access or judge model.
Pinned API inspected 2026-10-18:
https://github.com/jpezzulli/sglang-rtxpro6000/blob/d00d88efc8d6281b12be4f4073126aec95038c55/python/sglang/srt/entrypoints/openai/protocol.py
https://github.com/jpezzulli/sglang-rtxpro6000/blob/d00d88efc8d6281b12be4f4073126aec95038c55/python/sglang/srt/entrypoints/openai/serving_chat.py

SSE chunks are not tokens. Streaming token IDs/meta_info are unsupported by this
pinned chat endpoint. Usage and sglext are separate final chunks and are retained.
"""
from __future__ import annotations

import argparse
import ast
from concurrent.futures import ThreadPoolExecutor
from copy import deepcopy
from datetime import datetime, timezone
import hashlib
import http.client
import ipaddress
import json
import math
from pathlib import Path
import re
import socket
import statistics
import threading
import time
from urllib.parse import urlsplit

PHASE_ORDER = ["bf16_a", "fp8", "bf16_b"]
SOURCE_COMMIT = "d00d88efc8d6281b12be4f4073126aec95038c55"
PAYLOAD_KEYS = ("messages", "tools", "tool_choice", "chat_template_kwargs")


def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)


def value_hash(value):
    return hashlib.sha256(canonical(value).encode()).hexdigest()


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def dump(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n")


def finite_json(value):
    if isinstance(value, float):
        return math.isfinite(value)
    if isinstance(value, dict):
        return all(finite_json(v) for v in value.values())
    if isinstance(value, list):
        return all(finite_json(v) for v in value)
    return True


def strict_json(text):
    def bad(value):
        raise ValueError("Nonfinite JSON constant: " + value)
    return json.loads(text, parse_constant=bad)


def sample_public_requests(paths, count=10):
    """First/middle requests per ordered log, then 3/4, 1/4, 1/8, etc.

    For four logs this selects first+middle from each, then 3/4 from the first
    two. Exact path/line/hash and the source request object are retained. The
    diagnostic log does not contain original sampling fields; benchmark fields
    are explicitly added later. System/messages/tools are never truncated.
    """
    pools = []
    for path in paths:
        path = Path(path)
        requests = []
        with path.open() as f:
            for line_no, line in enumerate(f, 1):
                obj = strict_json(line)
                if obj.get("event") == "request":
                    requests.append((line_no, obj))
        if not requests:
            raise ValueError(f"No requests in {path}")
        pools.append((path, file_hash(path), requests))
    if not pools:
        return []
    selected, used = [], set()
    fractions = (0, .5, .75, .25, .125, .875, .375, .625)
    for fraction in fractions:
        for path, sha, requests in pools:
            index = min(len(requests) - 1, int(len(requests) * fraction))
            line_no, original = requests[index]
            key = (str(path.resolve()), line_no)
            if key in used:
                continue
            used.add(key)
            selected.append({"payload": {k: deepcopy(original[k]) for k in PAYLOAD_KEYS if k in original},
                             "source": {"path": str(path.resolve()), "sha256": sha, "line": line_no,
                                        "request_ordinal_1based": index + 1, "request_count_in_log": len(requests),
                                        "source_request_sha256": value_hash(original)},
                             "original_request": original})
            if len(selected) == count:
                return selected
    raise ValueError(f"Only {len(selected)} distinct requests available; {count} required")


def _quality_cases():
    cases = []
    tool = {"type": "function", "function": {"name": "submit_observation",
        "description": "Return the requested observed cells, without executing any actions.",
        "parameters": {"type": "object", "properties": {"cells": {"type": "array", "items": {"type": "array", "items": {"type": "integer"}}},
                        "count": {"type": "integer"}}, "required": ["cells", "count"], "additionalProperties": False}}}
    for i in range(10):
        # Public synthetic characters and independent exact answers. Holes and
        # disconnected cells matter; no ARC hidden/environment data are used.
        rows = [["." for _ in range(8)] for _ in range(6)]
        for r in range(1, 5):
            for c in range(1, 7):
                if (r in (1, 4) or c in (1, 6)) and (r + 2 * c + i) % 5:
                    rows[r][c] = "B"
        rows[i % 6][(i * 3) % 8] = "Y"
        r0, c0 = 10 + i, 20 + 2 * i
        expected_cells = [[r0 + r, c0 + c] for r in range(6) for c in range(8) if rows[r][c] == "B"]
        grid = "\n".join("".join(row) for row in rows)
        prompt = (f"Observe this 6 by 8 character crop with global origin row={r0}, col={c0}.\n{grid}\n"
                  "Return every cell whose literal character is B, as global [row,col] pairs sorted by row then col. "
                  "Preserve holes; Y and . are not B. Use submit_observation exactly once with cells and count. "
                  "Do not call Python, move anything, fill the bounding box, or provide prose.")
        cases.append({"id": f"quality_{i:02d}", "kind": "synthetic_exact_tool", "source": {"kind": "deterministic_public_synthetic", "version": 1},
                      "payload": {"messages": [{"role": "system", "content": "Read literal observations carefully and return the requested tool arguments."},
                                               {"role": "user", "content": prompt}], "tools": [tool], "tool_choice": "auto",
                                  "chat_template_kwargs": {"enable_thinking": False, "preserve_thinking": True}},
                      "expected": {"cells": expected_cells, "count": len(expected_cells)}})
    return cases


def make_suite(public_request_logs=None, *, output_tokens=256, seed=190018, model="flashnext"):
    """Pure offline suite. Pass four ordered baseline *_requests.jsonl paths.

    Ten immutable public snapshots are used when supplied; otherwise synthetic
    long-prefix requests provide an offline smoke suite (explicitly different).
    Every phase uses the same suite; never regenerate it after viewing outputs.
    """
    if not isinstance(output_tokens, int) or not 32 <= output_tokens <= 2048:
        raise ValueError("output_tokens must be a frozen integer in [32,2048]")
    public = sample_public_requests(public_request_logs or [], 10)
    tasks = []
    for i in range(10):
        if public:
            payload, source = deepcopy(public[i]["payload"]), public[i]["source"]
        else:
            prefix = "\n".join(f"Cell {n:04d}: row={n // 31}, column={n % 31}, code={(n * 7 + i) % 16}." for n in range(350 + 50 * (i % 3)))
            payload = {"messages": [{"role": "system", "content": "Reason using only the supplied public synthetic table."},
                                    {"role": "user", "content": prefix + "\nExplain the coordinate and code pattern and give worked examples."}],
                       "chat_template_kwargs": {"enable_thinking": True, "preserve_thinking": True}}
            source = {"kind": "public_synthetic_long_prefix", "case": i}
        tasks.append({"id": f"load_{i:02d}", "kind": "public_snapshot_fixed_length" if public else "synthetic_fixed_length",
                      "payload": payload, "source": source})
    latency = []
    for i in range(2):
        case = deepcopy(tasks[i]); case["id"] = f"latency_{i:02d}"
        latency.append(case)
    all_cases = latency + tasks + _quality_cases()
    for i, case in enumerate(all_cases):
        quality = case["kind"] == "synthetic_exact_tool"
        p = case["payload"]
        limit = 512 if quality else min(128, output_tokens) if case["id"].startswith("latency_") else output_tokens
        p.update(model=model, temperature=0.0, top_p=1.0, top_k=-1, seed=seed + i, n=1,
                 max_tokens=limit, min_tokens=0 if quality else limit, ignore_eos=not quality,
                 stream=True, stream_options={"include_usage": True, "continuous_usage_stats": True},
                 return_spec_tokens_details=True, return_cached_tokens_details=True,
                 cache_salt="arc3-019-v1-" + case["id"])
        case["payload_sha256"] = value_hash(p)
        case["fixed_output_tokens"] = None if quality else limit
    groups = [
        {"id": "latency_cold", "concurrency": 1, "cache": "cold", "flush_before": True, "task_ids": [c["id"] for c in latency]},
        {"id": "latency_hot", "concurrency": 1, "cache": "hot", "flush_before": False, "task_ids": [c["id"] for c in latency]},
        {"id": "load10_cold", "concurrency": 10, "cache": "cold", "flush_before": True, "task_ids": [c["id"] for c in tasks]},
        {"id": "load10_hot", "concurrency": 10, "cache": "hot", "flush_before": False, "task_ids": [c["id"] for c in tasks]},
        {"id": "quality10", "concurrency": 10, "cache": "isolated_salt_unprimed", "flush_before": False, "task_ids": [f"quality_{i:02d}" for i in range(10)]},
    ]
    suite = {"schema_version": 1, "id": "019-ple-fixed-requests-v1", "phase_order": PHASE_ORDER,
             "server_random_seed": seed, "source_api_commit": SOURCE_COMMIT, "cases": all_cases, "groups": groups,
             "public_sources": [{"source": p["source"], "original_request": p["original_request"]} for p in public],
             "controls": {"sampling": "temperature0/top_p1/top_k-1; fixed per-case seed in all phases",
                 "cache": "Unique per-case cache_salt; cold batch after backend-confirmed flush; hot repeats byte-identical payload after all cold requests finish.",
                 "public_snapshot_changes": "Preserve full messages/tools/tool_choice/template kwargs; add fixed benchmark generation/stream/cache fields. Original logs omit original sampling fields.",
                 "output_length": "Forced fixed-length throughput is not a quality task. Quality uses natural EOS, thinking off, a frozen 512-token cap.",
                 "quality_scope": "Ten deterministic synthetic literal-reading/tool tasks only; no full-game quality, hidden performance, judge model or generated-code execution.",
                 "streaming": "No return_token_ids or return_meta_info: pinned streaming endpoint rejects them. Chunks are not tokens."}}
    suite["suite_sha256"] = value_hash(suite)
    validate_suite(suite)
    return suite


def validate_suite(suite):
    content = {k: v for k, v in suite.items() if k != "suite_sha256"}
    if value_hash(content) != suite.get("suite_sha256"):
        raise ValueError("Suite hash mismatch")
    if suite["phase_order"] != PHASE_ORDER:
        raise ValueError("Frozen BF16/FP8/BF16 order required")
    cases = {c["id"]: c for c in suite["cases"]}
    if len(cases) != len(suite["cases"]):
        raise ValueError("Duplicate task IDs")
    for case in cases.values():
        p = case["payload"]
        if value_hash(p) != case["payload_sha256"] or not finite_json(p):
            raise ValueError("Corrupt/nonfinite payload")
        if not p.get("stream") or p.get("n") != 1 or p.get("return_token_ids") or p.get("return_meta_info") or p.get("return_prompt_token_ids"):
            raise ValueError("Unsupported streaming configuration")
        for message in p["messages"]:
            parts = message.get("content")
            if isinstance(parts, list):
                for part in parts:
                    for key in ("image_url", "audio_url", "video_url"):
                        if key in part:
                            value = part[key]
                            url = value.get("url", "") if isinstance(value, dict) else value
                            if not isinstance(url, str) or not url.startswith("data:"):
                                raise ValueError("Replay media must be inline data, never remote URLs")
    for group in suite["groups"]:
        if group["concurrency"] not in (1, 10) or not all(t in cases for t in group["task_ids"]):
            raise ValueError("Invalid frozen group")
    return {"valid": True, "suite_sha256": suite["suite_sha256"], "cases": len(cases),
            "total_requests_per_phase": sum(len(g["task_ids"]) for g in suite["groups"]),
            "public_request_count": len(suite["public_sources"])}


def _endpoint(base_url):
    p = urlsplit(base_url)
    host = "127.0.0.1" if p.hostname == "localhost" else p.hostname
    try:
        local = ipaddress.ip_address(host).is_loopback
    except (ValueError, TypeError):
        local = False
    if p.scheme != "http" or not local or p.username or p.password or p.query or p.fragment or p.path.rstrip("/") not in ("", "/v1"):
        raise ValueError("Only plain HTTP loopback service roots or /v1 are supported")
    return host, p.port or 80


class Ledger:
    def __init__(self, path):
        self.path = Path(path)
        self.lock = threading.Lock()
        self.sequence = 0

    def append(self, row):
        with self.lock:
            self.sequence += 1
            value = {"ledger_seq": self.sequence, **row}
            with self.path.open("a") as stream:
                stream.write(json.dumps(value, ensure_ascii=False, allow_nan=False) + "\n")
            return self.sequence


def _local_request(base_url, endpoint, *, method="GET", payload=None, timeout=5):
    host, port = _endpoint(base_url)
    connection = http.client.HTTPConnection(host, port, timeout=max(.1, timeout))
    captured_socket = [None]
    timer = _connection_deadline(connection, captured_socket, max(.1, timeout))
    try:
        body = canonical(payload).encode() if payload is not None else b"" if method == "POST" else None
        connection.request(method, endpoint, body=body, headers={"Content-Type": "application/json"})
        captured_socket[0] = connection.sock
        response = connection.getresponse()
        raw = response.read().decode("utf-8", errors="replace")
        return {"http_status": response.status, "body": raw}
    finally:
        timer.cancel()
        connection.close()


def _connection_deadline(connection, captured_socket, seconds):
    # Socket read timeouts reset when bytes trickle in; shutdown at an absolute
    # wall duration also interrupts a partial SSE line or slow response body.
    # Retain the socket because HTTPConnection clears .sock on Connection:close
    # while HTTPResponse can still own its file object.
    def terminate():
        sock = connection.sock or captured_socket[0]
        if sock is not None:
            try: sock.shutdown(socket.SHUT_RDWR)
            except OSError: pass
        connection.close()
    timer = threading.Timer(seconds, terminate)
    timer.daemon = True
    timer.start()
    return timer


def _sanitize_info(raw):
    # Public benchmark has no auth, but do not serialize server credentials if
    # an operator happens to configure them. The redacted response is labeled.
    try:
        value = strict_json(raw)
    except ValueError:
        return raw
    def clean(obj):
        if isinstance(obj, dict):
            return {k: "[REDACTED]" if any(t in k.lower() for t in ("api_key", "password", "secret")) and v else clean(v) for k, v in obj.items()}
        return [clean(v) for v in obj] if isinstance(obj, list) else obj
    return clean(value)


def _snapshot(base_url, endpoint, ledger, label, deadline):
    start = time.monotonic()
    try:
        if start >= deadline:
            raise TimeoutError("Phase deadline reached before diagnostics")
        result = _local_request(base_url, endpoint, timeout=min(3, deadline - start))
        if endpoint in ("/get_server_info", "/server_info"):
            result["body"] = _sanitize_info(result["body"])
            result["credentials_redacted"] = True
    except Exception as exc:
        result = {"error": type(exc).__name__, "detail": str(exc)}
    row = {"event": "server_snapshot", "label": label, "endpoint": endpoint,
           "epoch": time.time(), "duration_seconds": time.monotonic() - start, **result}
    ledger.append(row)
    return row


def _flush(base_url, serve_log, ledger, deadline, label):
    path = Path(serve_log) if serve_log is not None else None
    offset = path.stat().st_size if path and path.is_file() else None
    row = {"event": "cache_flush", "label": label, "epoch": time.time(), "serve_log_offset_before": offset}
    try:
        row.update(_local_request(base_url, "/flush_cache", method="POST", timeout=min(10, max(.1, deadline - time.monotonic()))))
        confirmed = False
        until = min(deadline, time.monotonic() + 10)
        while offset is not None and time.monotonic() < until:
            with path.open("rb") as f:
                f.seek(offset); recent = f.read()
            if b"Cache flushed successfully!" in recent:
                confirmed = True
                row["backend_confirmation_text"] = recent.decode(errors="replace")
                break
            time.sleep(.1)
        row["backend_confirmed"] = confirmed
    except Exception as exc:
        row.update(error=type(exc).__name__, detail=str(exc), backend_confirmed=False)
    ledger.append(row)
    if row.get("http_status") != 200 or not row["backend_confirmed"]:
        raise RuntimeError("Cold-cache group lacks new backend flush confirmation")
    return row


class StreamAccumulator:
    """Pure SSE decoder; callable on retained CPU fixtures without a server."""
    def __init__(self):
        self.content = ""
        self.reasoning = ""
        self.tools = {}
        self.usage = None
        self.sglext = {}
        self.finish_reason = None
        self.done = False
        self.errors = []
        self.first_any = self.first_visible = self.first_arguments = self.last_any = None
        self.first_payload_usage = None
        self.payload_events = 0

    def feed(self, data, elapsed):
        if data == "[DONE]":
            self.done = True
            return
        try:
            obj = strict_json(data)
        except ValueError as exc:
            self.errors.append("invalid_sse_json: " + str(exc)); return
        if not finite_json(obj):
            self.errors.append("nonfinite_sse_json"); return
        if obj.get("error"):
            self.errors.append(obj["error"])
        if obj.get("usage") is not None:
            self.usage = obj["usage"]
        if obj.get("sglext"):
            self.sglext.update(obj["sglext"])
        for choice in obj.get("choices", []):
            if choice.get("index", 0) != 0:
                self.errors.append("unexpected_choice_index"); continue
            self.finish_reason = choice.get("finish_reason") or self.finish_reason
            delta = choice.get("delta") or {}
            content = delta.get("content") or ""
            reasoning = delta.get("reasoning_content") or delta.get("reasoning") or ""
            if not isinstance(content, str) or not isinstance(reasoning, str):
                self.errors.append("nonstring_text_delta"); continue
            self.content += content; self.reasoning += reasoning
            arguments = False
            tool_payload = False
            for item in delta.get("tool_calls") or []:
                index = item.get("index", 0)
                target = self.tools.setdefault(index, {"id": "", "type": "function", "function": {"name": "", "arguments": ""}})
                if item.get("id"):
                    target["id"] = item["id"]
                fn = item.get("function") or {}
                for key in ("name", "arguments"):
                    fragment = fn.get(key) or ""
                    if not isinstance(fragment, str):
                        self.errors.append("nonstring_tool_delta"); continue
                    target["function"][key] += fragment
                    tool_payload |= bool(fragment)
                    if key == "arguments": arguments |= bool(fragment)
            if content and self.first_visible is None: self.first_visible = elapsed
            if arguments and self.first_arguments is None: self.first_arguments = elapsed
            if content or reasoning or tool_payload:
                if self.first_any is None:
                    self.first_any = elapsed
                    self.first_payload_usage = (obj.get("usage") or {}).get("completion_tokens")
                self.last_any = elapsed
                self.payload_events += 1

    def result(self):
        return {"message": {"content": self.content, "reasoning_content": self.reasoning,
                            "tool_calls": [self.tools[k] for k in sorted(self.tools)]},
                "usage": self.usage, "sglext": self.sglext, "finish_reason": self.finish_reason,
                "sse_done": self.done, "stream_errors": self.errors,
                "first_semantic_payload_seconds": self.first_any, "first_visible_content_seconds": self.first_visible,
                "first_tool_arguments_seconds": self.first_arguments, "last_semantic_payload_seconds": self.last_any,
                "first_payload_completion_tokens_if_reported": self.first_payload_usage,
                "semantic_payload_events_not_tokens": self.payload_events}


def response_checks(case, result):
    message = result["message"]
    parsed, format_errors = [], []
    for tool in message["tool_calls"]:
        try:
            args = strict_json(tool["function"]["arguments"])
            if not finite_json(args): raise ValueError("nonfinite arguments")
            if tool["function"]["name"] == "python":
                if not isinstance(args, dict) or not isinstance(args.get("code"), str):
                    raise ValueError("python tool has no string code")
                ast.parse(args["code"])  # Syntax only. Never executes code.
            parsed.append({"name": tool["function"]["name"], "arguments": args})
        except (ValueError, SyntaxError, TypeError) as exc:
            format_errors.append(str(exc))
    quality = case["kind"] == "synthetic_exact_tool"
    arg = parsed[0]["arguments"] if len(parsed) == 1 else None
    integer_cells = (isinstance(arg, dict) and type(arg.get("count")) is int
                     and isinstance(arg.get("cells"), list) and all(isinstance(cell, list) and len(cell) == 2
                     and all(type(coord) is int for coord in cell) for cell in arg["cells"]))
    exact = (len(parsed) == 1 and parsed[0]["name"] == "submit_observation" and integer_cells
             and arg == case.get("expected") and not format_errors and not message["content"].strip())
    return {"tool_calls": len(message["tool_calls"]), "parsed_tool_arguments": parsed,
            "format_errors": format_errors, "content_contains_replacement_character": "\ufffd" in canonical(message),
            "quality_exact_match": bool(exact) if quality else None,
            "quality_truncated": result.get("finish_reason") == "length" if quality else None,
            "public_format_observation_only": not quality,
            "note": "Fixed-length replay may cut a tool call intentionally. Format observations there are not quality failures. No generated Python is executed; AST parse checks syntax only."}


def _stream_request(base_url, case, group, phase, ledger, deadline, request_timeout=120):
    start, epoch = time.monotonic(), time.time()
    end = min(deadline, start + request_timeout)
    identity = {"phase": phase, "group": group, "case_id": case["id"], "payload_sha256": case["payload_sha256"]}
    ledger.append({"event": "request", **identity, "epoch": epoch, "payload": case["payload"]})
    accumulator, error, status = StreamAccumulator(), None, None
    connection = None
    timer = None
    captured_socket = [None]
    headers_time = None
    try:
        if start >= end: raise TimeoutError("No request budget remains")
        host, port = _endpoint(base_url)
        connection = http.client.HTTPConnection(host, port, timeout=max(.1, end - start))
        timer = _connection_deadline(connection, captured_socket, max(.1, end - time.monotonic()))
        connection.request("POST", "/v1/chat/completions", body=canonical(case["payload"]).encode(), headers={"Content-Type": "application/json"})
        captured_socket[0] = connection.sock
        response = connection.getresponse()
        status, headers_time = response.status, time.monotonic() - start
        ledger.append({"event": "response_headers", **identity, "elapsed_seconds": headers_time,
                       "http_status": status, "headers": dict(response.getheaders())})
        if status != 200:
            body = response.read().decode(errors="replace")
            ledger.append({"event": "http_error_body", **identity, "body": body})
            raise RuntimeError(f"HTTP {status}")
        data_lines = []
        while True:
            remaining = end - time.monotonic()
            if remaining <= 0: raise TimeoutError("Absolute request/phase deadline")
            if connection.sock is not None:
                connection.sock.settimeout(remaining)
            line = response.readline()
            if not line: break
            elapsed = time.monotonic() - start
            raw = line.decode("utf-8", errors="replace").rstrip("\r\n")
            ledger.append({"event": "sse_line", **identity, "elapsed_seconds": elapsed, "data": raw})
            if raw.startswith("data:"):
                data_lines.append(raw[5:].lstrip(" "))
            elif raw == "" and data_lines:
                accumulator.feed("\n".join(data_lines), elapsed)
                data_lines = []
                if accumulator.done: break
        if data_lines: accumulator.feed("\n".join(data_lines), time.monotonic() - start)
        if not accumulator.done: raise RuntimeError("SSE ended without [DONE]")
        if time.monotonic() > end: raise TimeoutError("Completion observed after absolute request/phase deadline")
    except Exception as exc:
        error = {"type": type(exc).__name__, "detail": str(exc)}
    finally:
        if timer: timer.cancel()
        if connection: connection.close()
    duration = time.monotonic() - start
    result = {**identity, "started_epoch": epoch, "elapsed_seconds": duration, "headers_seconds": headers_time,
              "http_status": status, "error": error, **accumulator.result()}
    usage = result.get("usage") or {}
    count = usage.get("completion_tokens")
    interval = (result["last_semantic_payload_seconds"] - result["first_semantic_payload_seconds"]
                if result["first_semantic_payload_seconds"] is not None and result["last_semantic_payload_seconds"] is not None else None)
    first_count = result["first_payload_completion_tokens_if_reported"]
    result["post_first_payload_usage_tokens_per_second_proxy"] = ((count - first_count) / interval
        if isinstance(count, int) and isinstance(first_count, int) and interval and interval > 0 and count >= first_count else None)
    result["usage_output_tokens_per_end_to_end_second"] = count / duration if isinstance(count, int) and duration > 0 else None
    result["fixed_output_length_matches"] = count == case["fixed_output_tokens"] if case["fixed_output_tokens"] is not None else None
    result["checks"] = response_checks(case, result)
    result["complete"] = error is None and result["sse_done"] and not result["stream_errors"] and isinstance(count, int)
    ledger.append({"event": "response_complete", **result})
    return result


def _stats(values):
    values = [v for v in values if isinstance(v, (int, float)) and math.isfinite(v)]
    if not values: return {"n": 0, "mean": None, "median": None}
    return {"n": len(values), "mean": statistics.fmean(values), "median": statistics.median(values),
            "min": min(values), "max": max(values)}


def _group_summary(group, rows, elapsed, flush):
    tokens = sum((r.get("usage") or {}).get("completion_tokens", 0) or 0 for r in rows)
    cached = [((r.get("usage") or {}).get("prompt_tokens_details") or {}).get("cached_tokens") for r in rows]
    proposed = correct = 0
    spec_records = 0
    for r in rows:
        d = (r.get("sglext") or {}).get("spec_tokens_details")
        if isinstance(d, dict):
            spec_records += 1
            proposed += d.get("spec_num_proposed_drafts", 0)
            correct += d.get("spec_num_correct_drafts", 0)
    return {**group, "requests_expected": len(group["task_ids"]), "requests_completed": sum(r["complete"] for r in rows),
            "batch_wall_seconds": elapsed, "completion_tokens": tokens,
            "aggregate_output_tokens_per_batch_wall_second": tokens / elapsed if elapsed else None,
            "first_semantic_payload_seconds": _stats([r["first_semantic_payload_seconds"] for r in rows]),
            "first_visible_content_seconds": _stats([r["first_visible_content_seconds"] for r in rows]),
            "first_tool_arguments_seconds": _stats([r["first_tool_arguments_seconds"] for r in rows]),
            "post_first_payload_token_rate_proxy": _stats([r["post_first_payload_usage_tokens_per_second_proxy"] for r in rows]),
            "end_to_end_seconds": _stats([r["elapsed_seconds"] for r in rows]),
            "fixed_lengths_all_match": all(r["fixed_output_length_matches"] is True for r in rows) if group["id"] != "quality10" else None,
            "cached_tokens": cached, "cache_usage_all_known": all(isinstance(n, int) for n in cached),
            "cache_condition_observed": all(n == 0 for n in cached) if group["cache"] == "cold" else all(isinstance(n, int) and n > 0 for n in cached) if group["cache"] == "hot" else None,
            "flush": flush, "quality_exact_matches": sum(r["checks"]["quality_exact_match"] is True for r in rows) if group["id"] == "quality10" else None,
            "quality_truncations": sum(r["checks"]["quality_truncated"] is True for r in rows) if group["id"] == "quality10" else None,
            "spec_details_requests": spec_records, "spec_proposed_drafts": proposed if spec_records else None,
            "spec_correct_drafts": correct if spec_records else None,
            "spec_accept_rate_weighted": correct / proposed if proposed else None,
            "all_requests_complete": len(rows) == len(group["task_ids"]) and all(r["complete"] for r in rows)}


def run_phase(base_url, outdir, suite, deadline, *, phase, serve_log=None, metadata=None, request_timeout=120, metrics_interval=5):
    """Explicit network operation against an already-ready loopback server.

    deadline is an ABSOLUTE time.monotonic() deadline; caller owns service
    lifecycle/GPU/RSS measurement and should reserve shutdown time separately.
    This function refuses reuse of its own output files and never shortens the
    frozen suite. The caller's serve.log/launcher-state may already be present.
    """
    validate_suite(suite); _endpoint(base_url)
    if phase not in PHASE_ORDER: raise ValueError("Unknown BF16/FP8 phase")
    if deadline <= time.monotonic(): raise ValueError("Expired monotonic deadline")
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    owned = ("suite.json", "manifest.json", "phase-report.json", "responses.jsonl", "server-metrics.jsonl")
    collisions = [name for name in owned if (outdir / name).exists()]
    if collisions: raise FileExistsError("Refusing to overwrite microbenchmark outputs: " + ", ".join(collisions))
    ledger, diagnostics = Ledger(outdir / "responses.jsonl"), Ledger(outdir / "server-metrics.jsonl")
    dump(outdir / "suite.json", suite)
    manifest = {"schema_version": 1, "phase": phase, "suite_sha256": suite["suite_sha256"],
                "module_sha256": file_hash(__file__), "started_utc": datetime.now(timezone.utc).isoformat(),
                "started_epoch": time.time(), "base_url": base_url, "deadline_monotonic": deadline,
                "metadata": metadata or {}, "status": "running", "groups": [], "requests": [],
                "semantics": {"ttft": "First semantic SSE payload, including reasoning/tool payload. Parser buffering can delay it beyond actual first generated token.",
                    "decode": "Continuous usage delta divided by first-to-last payload interval is a proxy, not a GPU kernel rate. Null if usage is missing at first payload.",
                    "batch_rate": "Successful-return completion tokens divided by actual group wall duration; retain failures and incomplete groups.",
                    "quality": "Exact synthetic answers only. Fixed-length public replays are deliberately truncated and cannot establish full-game quality.",
                    "numerics": "JSON/usage/output structural checks cannot prove hidden logits or model weights are finite; no hidden-state instrumentation."}}
    dump(outdir / "manifest.json", manifest)
    stop, current_group = threading.Event(), ["initial"]
    def sample():
        while not stop.wait(metrics_interval) and time.monotonic() < deadline:
            _snapshot(base_url, "/metrics", diagnostics, current_group[0], deadline)
    sampler = threading.Thread(target=sample, daemon=True)
    start = time.monotonic()
    try:
        info = _snapshot(base_url, "/get_server_info", diagnostics, "start", deadline)
        manifest["server_info_start"] = info
        sampler.start()
        cases = {c["id"]: c for c in suite["cases"]}
        for group in suite["groups"]:
            if time.monotonic() >= deadline: raise TimeoutError("Phase deadline before next frozen group")
            current_group[0] = group["id"]
            flush = _flush(base_url, serve_log, diagnostics, deadline, group["id"]) if group["flush_before"] else None
            _snapshot(base_url, "/metrics", diagnostics, group["id"] + ":before", deadline)
            group_start = time.monotonic()
            with ThreadPoolExecutor(max_workers=group["concurrency"]) as pool:
                futures = [pool.submit(_stream_request, base_url, cases[tid], group["id"], phase, ledger, deadline, request_timeout) for tid in group["task_ids"]]
                rows = [f.result() for f in futures]
            summary = _group_summary(group, rows, time.monotonic() - group_start, flush)
            manifest["requests"].extend(rows); manifest["groups"].append(summary)
            _snapshot(base_url, "/metrics", diagnostics, group["id"] + ":after", deadline)
            dump(outdir / "manifest.json", manifest)
            if not summary["all_requests_complete"]:
                raise RuntimeError("Frozen group incomplete; aborting instead of comparing a subset")
        manifest["status"] = "complete"
    except Exception as exc:
        manifest["status"] = "incomplete"
        manifest["error"] = {"type": type(exc).__name__, "detail": str(exc)}
    finally:
        stop.set()
        if sampler.is_alive(): sampler.join(timeout=4)
        manifest["metrics_sampler_stopped"] = not sampler.is_alive()
        manifest["finished_epoch"] = time.time()
        manifest["elapsed_seconds"] = time.monotonic() - start
        manifest["raw_response_sha256"] = file_hash(ledger.path) if ledger.path.exists() else None
        manifest["server_metrics_sha256"] = file_hash(diagnostics.path) if diagnostics.path.exists() else None
        dump(outdir / "manifest.json", manifest)
        dump(outdir / "phase-report.json", manifest)
    return manifest


def compare_phase_reports(paths):
    """Pure offline comparison; incomplete groups cannot produce speed claims."""
    reports = [json.loads(Path(p).read_text()) for p in paths]
    by_phase = {r["phase"]: r for r in reports}
    if len(reports) != 3 or set(by_phase) != set(PHASE_ORDER):
        raise ValueError("Exactly bf16_a, fp8, bf16_b reports required")
    ordered = [by_phase[p] for p in PHASE_ORDER]
    suite_same = len({r["suite_sha256"] for r in ordered}) == 1
    configs = [(r.get("server_info_start") or {}).get("body") for r in ordered]
    config_fields = ("dtype", "quantization", "kv_cache_dtype", "context_length", "max_running_requests",
                     "chunked_prefill_size", "max_prefill_tokens", "page_size", "mem_fraction_static",
                     "schedule_policy", "random_seed", "speculative_algorithm", "speculative_num_steps",
                     "speculative_num_draft_tokens", "speculative_accept_threshold_single",
                     "speculative_accept_threshold_acc", "mamba_ssm_dtype", "mamba_radix_cache_strategy",
                     "gdn_mtp_cache_mode", "reasoning_parser", "tool_call_parser", "default_chat_template_kwargs")
    configs_present = all(isinstance(c, dict) for c in configs)
    config_checks = {key: configs_present and all(key in c for c in configs)
                    and configs[0][key] == configs[1][key] == configs[2][key] for key in config_fields}
    chronological = all(a["finished_epoch"] <= b["started_epoch"] for a, b in zip(ordered, ordered[1:]))
    raw_checks = []
    for path in paths:
        r = json.loads(Path(path).read_text())
        raw = Path(path).parent / "responses.jsonl"
        raw_checks.append({"phase": r["phase"], "path": str(raw.resolve()),
                           "exists": raw.is_file(), "hash_matches": raw.is_file() and file_hash(raw) == r.get("raw_response_sha256")})
    provenance_valid = all(config_checks.values()) and chronological and all(r["hash_matches"] for r in raw_checks)
    maps = [{g["id"]: g for g in r["groups"]} for r in ordered]
    rows = []
    for group in sorted(set().union(*(set(m) for m in maps))):
        gs = [m.get(group) for m in maps]
        compatible = suite_same and provenance_valid and all(g and g["all_requests_complete"] for g in gs)
        if group != "quality10":
            compatible = compatible and all(g["fixed_lengths_all_match"] and g["cache_condition_observed"] is True for g in gs if g)
        row = {"group": group, "compatible_complete_group": bool(compatible),
               "phases": {phase: g for phase, g in zip(PHASE_ORDER, gs)}}
        if compatible and group != "quality10":
            aa, bb, cc = [g["aggregate_output_tokens_per_batch_wall_second"] for g in gs]
            baseline = (aa + cc) / 2
            row["fp8_rate_vs_mean_bf16_ratio"] = bb / baseline if baseline else None
            row["bf16_repeat_rate_ratio"] = cc / aa if aa else None
            row["fp8_rate_above_both_bf16_repeats"] = bb > max(aa, cc)
        rows.append(row)
    request_maps = [{(r["group"], r["case_id"]): r for r in report["requests"]} for report in ordered]
    paired = []
    for key in sorted(set().union(*(set(m) for m in request_maps))):
        rs = [m.get(key) for m in request_maps]
        same = all(rs) and len({r["payload_sha256"] for r in rs}) == 1
        def semantic(r):
            if r is None: return None
            msg = r["message"]
            return {"content": msg["content"], "reasoning_content": msg["reasoning_content"],
                    "tool_functions": [t["function"] for t in msg["tool_calls"]]}
        paired.append({"group": key[0], "case_id": key[1], "same_request_payload": bool(same),
                       "bf16_repeat_exact_output": semantic(rs[0]) == semantic(rs[2]) if all(rs) else None,
                       "fp8_equals_bf16_a": semantic(rs[1]) == semantic(rs[0]) if all(rs) else None,
                       "quality_exact_matches": {p: r["checks"]["quality_exact_match"] if r else None for p, r in zip(PHASE_ORDER, rs)}})
    return {"schema_version": 1, "phase_order": PHASE_ORDER, "same_suite": suite_same,
            "server_config_checks": config_checks, "phase_execution_order_verified": chronological,
            "raw_response_checks": raw_checks, "provenance_checks_pass": provenance_valid,
            "all_phases_complete": all(r["status"] == "complete" for r in ordered),
            "all_request_payloads_match": all(p["same_request_payload"] for p in paired),
            "sources": [{"path": str(Path(p).resolve()), "sha256": file_hash(p)} for p in paths],
            "groups": rows, "paired_requests": paired,
            "limits": ["Same seeds do not guarantee deterministic concurrent generation; BF16 repeat is a drift diagnostic, not a confidence interval.",
                       "Synthetic exact-answer quality is limited. Public forced-length outputs have no semantic judge and are never executed.",
                       "Client TTFT/decode proxies and aggregate throughput do not identify PLE kernel causality. Combine with independent weight/config/GPU/RSS audits.",
                       "Selected server controls and raw response hashes are checked here. Different model/tokenizer/draft paths require the runner's content/weight audit, not path-string equality.",
                       "No automatic acceptance, serving change, game run or submission follows this analysis."]}


def cpu_check():
    """No-network in-memory checks; no new tests/CI entry point."""
    suite = make_suite()
    check = validate_suite(suite)
    acc = StreamAccumulator()
    acc.feed('{"choices":[{"index":0,"delta":{"role":"assistant"}}]}', .1)
    assert acc.first_any is None
    chunks = [('{"choices":[{"index":0,"delta":{"reasoning_content":"r"}}],"usage":{"completion_tokens":1}}', .2),
              ('{"choices":[{"index":0,"delta":{"tool_calls":[{"index":0,"id":"x","function":{"name":"submit_observation","arguments":"{\\"cells\\":"}}]}}],"usage":{"completion_tokens":2}}', .3),
              ('{"choices":[{"index":0,"delta":{"tool_calls":[{"index":0,"function":{"arguments":"[],\\"count\\":0}"}}]},"finish_reason":"tool_calls"}]}', .4),
              ('{"choices":[],"usage":{"prompt_tokens":10,"completion_tokens":7,"total_tokens":17}}', .5),
              ('{"choices":[],"sglext":{"spec_tokens_details":{"spec_num_proposed_drafts":8,"spec_num_correct_drafts":6}}}', .6)]
    for data, t in chunks: acc.feed(data, t)
    acc.feed('[DONE]', .7)
    result = acc.result()
    assert result["usage"]["completion_tokens"] == 7 and result["first_semantic_payload_seconds"] == .2
    assert result["first_tool_arguments_seconds"] == .3 and result["first_visible_content_seconds"] is None
    assert strict_json(result["message"]["tool_calls"][0]["function"]["arguments"]) == {"cells": [], "count": 0}
    checks = response_checks({"kind": "synthetic_exact_tool", "expected": {"cells": [], "count": 0}}, result)
    assert checks["quality_exact_match"]
    bad = StreamAccumulator(); bad.feed('{"usage":{"completion_tokens":NaN}}', 0)
    assert bad.errors and not bad.done
    for url in ("https://127.0.0.1:8001", "http://example.com", "http://127.0.0.1.evil.test", "http://127.0.0.1:8001/anything"):
        try: _endpoint(url)
        except ValueError: pass
        else: raise AssertionError("Unsafe endpoint accepted")
    return {"passed": True, "suite": check, "checks": ["offline_suite_hash", "role_chunk_not_token", "split_tool_arguments", "separate_final_usage_and_sglext", "reasoning_vs_visible_vs_tool_latency", "strict_nonfinite_rejection", "loopback_only"],
            "network_used": False, "models_or_logged_code_executed": False}


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    sub = parser.add_subparsers(dest="command", required=True)
    prep = sub.add_parser("prepare")
    prep.add_argument("--request-log", action="append", default=[], type=Path)
    prep.add_argument("--output", type=Path, required=True)
    prep.add_argument("--output-tokens", type=int, default=256)
    prep.add_argument("--seed", type=int, default=190018)
    prep.add_argument("--model", default="flashnext")
    valid = sub.add_parser("validate"); valid.add_argument("suite", type=Path)
    sub.add_parser("cpu-check")
    run = sub.add_parser("run")
    run.add_argument("--allow-local-network", action="store_true")
    run.add_argument("--suite", type=Path, required=True); run.add_argument("--output-dir", type=Path, required=True)
    run.add_argument("--phase", choices=PHASE_ORDER, required=True); run.add_argument("--base-url", default="http://127.0.0.1:8001")
    run.add_argument("--budget-seconds", type=float, default=360); run.add_argument("--serve-log", type=Path, required=True)
    analyze = sub.add_parser("analyze"); analyze.add_argument("reports", nargs=3, type=Path); analyze.add_argument("--output", type=Path, required=True)
    args = parser.parse_args()
    if args.command == "prepare":
        result = make_suite(args.request_log, output_tokens=args.output_tokens, seed=args.seed, model=args.model)
        if args.output.exists(): raise FileExistsError("Refusing to overwrite a frozen suite")
        dump(args.output, result); print(json.dumps(validate_suite(result), indent=2))
    elif args.command == "validate": print(json.dumps(validate_suite(json.loads(args.suite.read_text())), indent=2))
    elif args.command == "cpu-check": print(json.dumps(cpu_check(), indent=2))
    elif args.command == "run":
        if not args.allow_local_network: parser.error("run requires explicit --allow-local-network")
        result = run_phase(args.base_url, args.output_dir, json.loads(args.suite.read_text()), time.monotonic() + args.budget_seconds,
                           phase=args.phase, serve_log=args.serve_log)
        print(json.dumps({k: result[k] for k in ("phase", "status", "elapsed_seconds")}, indent=2))
        if result["status"] != "complete": raise SystemExit(1)
    elif args.command == "analyze":
        result = compare_phase_reports(args.reports); dump(args.output, result)
        print(json.dumps({k: result[k] for k in ("same_suite", "all_phases_complete", "all_request_payloads_match")}, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/019-code/src/arc3_research/ple_session.py
"""Bounded, offline BF16 / FP8 PLE / BF16 serving experiment.

This runner never imports a game, unpickles a benchmark or executes model output.
Only an explicit CLI run launches the pinned serving program.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import os
from pathlib import Path
import signal
import subprocess
import sys
import threading
import time
import zipfile
from urllib.request import urlopen

from arc3_research.session_runner import start_gpu_sampling, stop_group


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def save(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n")
    temporary.replace(path)


def validate_config(config):
    expected = {"id": "019-fp8-ple", "account": "jiyujia",
                "phase_order": ["bf16_a", "fp8", "bf16_b"],
                "server_seed": 190018, "port": 8001,
                "hard_limit_seconds": 4800, "startup_limit_seconds": 900,
                "phase_budget_seconds": 360, "weight_audit_limit_seconds": 240}
    for key, value in expected.items():
        if config.get(key) != value:
            raise ValueError(f"Frozen 019 setting differs: {key}")
    if config.get("mode") not in {"weights_only", "microbench"}:
        raise ValueError("Unknown 019 execution mode")
    if set(config["inputs"]) != {"bf16", "fp8", "draft", "wheelhouse"}:
        raise ValueError("Expected exactly three model inputs and one wheelhouse")


def resolve_inputs(specs, root="/kaggle/input"):
    root, found = Path(root), {}
    for key, relative in specs.items():
        expected = root / relative
        marker = "requirements.lock" if key == "wheelhouse" else "config.json"
        if expected.is_dir() and (expected / marker).is_file():
            choices = [expected]
        else:
            slug = Path(relative).parts[2]
            choices = [p.parent for p in root.rglob(marker)
                       if slug in p.parts and (key == "wheelhouse" or p.parent.name == "1")]
        if len(choices) != 1:
            raise RuntimeError(f"019 {key}: expected one attached input, got {choices}")
        found[key] = str(choices[0].resolve())
        required = ["wheels", "hot_tokens_64k.pt"] if key == "wheelhouse" else ["model.safetensors.index.json"]
        if any(not (choices[0] / name).exists() for name in required):
            raise FileNotFoundError(f"019 incomplete {key} attachment")
    return found


def normalized_args(args):
    """Only path identity may differ; all effective command switches must match."""
    result = list(args)
    for flag in ("--model-path", "--chat-template", "--speculative-draft-model-path"):
        if flag in result:
            result[result.index(flag) + 1] = "<" + flag[2:] + ">"
    return result


def audit_wheel_sources(wheelhouse, expected):
    wheels = sorted((Path(wheelhouse) / "wheels").glob("sglang-*.whl"))
    if len(wheels) != 1:
        raise RuntimeError("Expected one pinned SGLang wheel")
    checks = {}
    with zipfile.ZipFile(wheels[0]) as archive:
        for relative, digest in expected.items():
            observed = hashlib.sha256(archive.read(relative)).hexdigest()
            checks[relative] = {"expected": digest, "observed": observed, "matches": observed == digest}
    return {"wheel": str(wheels[0]), "wheel_bytes": wheels[0].stat().st_size,
            "checks": checks, "passed": all(row["matches"] for row in checks.values())}


def host_sample(groups):
    sample = {"epoch": time.time(), "processes": [], "meminfo_kib": {}}
    try:
        for line in Path("/proc/meminfo").read_text().splitlines():
            key, value = line.split(":", 1)
            if key in {"MemTotal", "MemAvailable", "Mlocked", "Unevictable", "Cached"}:
                sample["meminfo_kib"][key] = int(value.split()[0])
        for directory in Path("/proc").iterdir():
            if not directory.name.isdigit():
                continue
            try:
                stat = (directory / "stat").read_text().rsplit(")", 1)[1].split()
                if int(stat[2]) not in groups:
                    continue
                fields = {}
                for line in (directory / "status").read_text().splitlines():
                    key, value = line.split(":", 1)
                    if key in {"VmRSS", "VmLck", "VmPin", "VmHWM"}:
                        fields[key + "_kib"] = int(value.split()[0])
                sample["processes"].append({"pid": int(directory.name), "pgid": int(stat[2]), **fields})
            except (OSError, ValueError, IndexError):
                continue
    except OSError as error:
        sample["unavailable"] = str(error)
    sample["note"] = "Per-process RSS may double-count shared pages; VmLck/VmPin do not prove the complete CUDA pinned allocation."
    return sample


def run(config_path, payload_root, output, input_root="/kaggle/input"):
    from arc3_research.ple_microbench import validate_suite, run_phase, compare_phase_reports

    payload_root, output = Path(payload_root), Path(output)
    config = json.loads(Path(config_path).read_text())
    validate_config(config)
    if os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").lower() in {"1", "true"}:
        raise RuntimeError("019 is a research microbench, never a competition submission")
    if output.exists() and any(output.iterdir()):
        raise FileExistsError("Use a fresh 019 output directory; never overwrite raw evidence")
    output.mkdir(parents=True, exist_ok=True)
    bundle = json.loads((payload_root / "payload-manifest.json").read_text())
    for relative, expected in bundle["sha256"].items():
        if sha(payload_root / relative) != expected:
            raise ValueError(f"019 payload hash mismatch: {relative}")
    suite = json.loads((payload_root / "suite.json").read_text())
    validate_suite(suite)
    paths = resolve_inputs(config["inputs"], input_root)
    save(output / "input-paths.json", paths)
    started = time.monotonic()
    report = {"config": config, "payload": bundle, "started_at_epoch": time.time(),
              "phases": [], "status": "initializing", "gpu_started": False,
              "competition_submission_created": False}
    done = threading.Event()
    drivers, groups = [], set()
    monitor = None

    def checkpoint():
        report["elapsed_seconds"] = time.monotonic() - started
        save(output / "session.json", report)

    def watchdog():
        if done.wait(config["hard_limit_seconds"] - 30):
            return
        save(output / "hard-limit.json", {"status": "hard_limit", "elapsed_seconds": time.monotonic() - started})
        for process in drivers:
            if process.poll() is None:
                try:
                    os.killpg(process.pid, signal.SIGTERM)
                except ProcessLookupError:
                    pass
        time.sleep(10)
        # These groups were registered by this session, never discovered by name.
        for pgid in list(groups):
            try:
                os.killpg(pgid, signal.SIGKILL)
            except ProcessLookupError:
                pass
        os._exit(124)

    def host_loop():
        while not done.is_set():
            row = host_sample(set(groups))
            with (output / "host-memory.jsonl").open("a") as stream:
                stream.write(json.dumps(row) + "\n")
            done.wait(10)

    threading.Thread(target=watchdog, daemon=True).start()
    sampler = threading.Thread(target=host_loop, daemon=True)
    sampler.start()
    try:
        checkpoint()
        runtime_audit = audit_wheel_sources(paths["wheelhouse"], config["runtime_source_sha256"])
        save(output / "wheel-source-check.json", runtime_audit)
        if not runtime_audit["passed"]:
            raise RuntimeError("Attached wheel source differs from the audited runtime; no serving")
        # Isolated CPU process imposes a real upper bound on attachment reads.
        command = [sys.executable, "-m", "arc3_research.ple_weights",
                   "--bf16-dir", paths["bf16"], "--fp8-dir", paths["fp8"],
                   "--draft-dir", paths["draft"], "--contract", str(payload_root / "weight-contract.json"),
                   "--output", str(output / "weight-audit.json")]
        with (output / "weight-audit.log").open("w") as stream:
            subprocess.run(command, stdout=stream, stderr=subprocess.STDOUT, check=True,
                           timeout=config["weight_audit_limit_seconds"])
        weight_audit = json.loads((output / "weight-audit.json").read_text())
        if weight_audit.get("passed") is not True:
            raise RuntimeError("Mounted weight contract failed; GPU server was not started")
        report["weight_audit_passed"] = True
        if config["mode"] == "weights_only":
            report["status"] = "cpu_preflight_complete"
            return report
        monitor = start_gpu_sampling(output / "gpu-utilization.jsonl", 10)
        common_args = None
        for phase in config["phase_order"]:
            if config["hard_limit_seconds"] - (time.monotonic() - started) < config["startup_limit_seconds"] + config["phase_budget_seconds"] + 60:
                raise RuntimeError("Insufficient remaining session budget for a complete phase; no shortened phase")
            directory = output / phase
            directory.mkdir()
            params = {"model": paths["fp8" if phase == "fp8" else "bf16"],
                      "draft": paths["draft"], "wheelhouse": paths["wheelhouse"],
                      "working_dir": str(directory), "port": config["port"],
                      "seed": config["server_seed"], "startup_limit_seconds": config["startup_limit_seconds"],
                      "runtime_source_sha256": config["runtime_source_sha256"]}
            save(directory / "launch-parameters.json", params)
            monitor.set_phase(phase + ":startup")
            phase_started = time.monotonic()
            entry = {"phase": phase, "started_at_epoch": time.time(), "status": "loading"}
            report["phases"].append(entry)
            checkpoint()
            with (directory / "launcher.log").open("w") as stream:
                driver = subprocess.Popen([sys.executable, str(payload_root / "launch_server.py"),
                                           str(directory / "launch-parameters.json")],
                                          stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
            drivers.append(driver)
            groups.add(driver.pid)
            state = {}
            try:
                while time.monotonic() - phase_started < config["startup_limit_seconds"]:
                    state_path = directory / "launcher-state.json"
                    if state_path.exists():
                        state = json.loads(state_path.read_text())
                        if state.get("server_pid"):
                            groups.add(state["server_pid"])
                            report["gpu_started"] = True
                        if state.get("status") == "ready":
                            break
                    if driver.poll() is not None:
                        raise RuntimeError(f"{phase} launcher exited: {driver.returncode}")
                    time.sleep(2)
                else:
                    raise TimeoutError(f"{phase} model startup exceeded frozen limit")
                entry["startup_seconds"] = time.monotonic() - phase_started
                entry["launch_state"] = state
                normalized = normalized_args(state["args"])
                if common_args is not None and normalized != common_args:
                    raise RuntimeError("Serving arguments changed beyond target/template/draft-view paths")
                common_args = normalized
                monitor.set_phase(phase + ":requests")
                entry["status"] = "benchmarking"
                checkpoint()
                phase_report = run_phase(f"http://127.0.0.1:{config['port']}", directory,
                                         suite, time.monotonic() + config["phase_budget_seconds"],
                                         phase=phase, serve_log=directory / "serve.log")
                entry["benchmark_report"] = phase_report
                if phase_report.get("status") != "complete":
                    raise RuntimeError("Frozen request suite incomplete; not starting another model")
                if any(g["id"] != "quality10" and (not g["fixed_lengths_all_match"] or g["cache_condition_observed"] is not True)
                       for g in phase_report["groups"]):
                    raise RuntimeError("Output-length/cache controls not established; not starting another model")
                entry["status"] = "requests_finished"
            finally:
                monitor.set_phase(phase + ":shutdown")
                stopped = stop_group(driver, grace=20)
                groups.discard(driver.pid)
                final_state_path = directory / "launcher-state.json"
                final_state = json.loads(final_state_path.read_text()) if final_state_path.exists() else {}
                server_pid = state.get("server_pid") or final_state.get("server_pid")
                if server_pid:
                    # Driver owns and drains this separate server group.
                    try:
                        os.killpg(server_pid, 0)
                        server_stopped = False
                    except ProcessLookupError:
                        server_stopped = True
                    if not server_stopped:
                        os.killpg(server_pid, signal.SIGKILL)
                        time.sleep(1)
                        raise RuntimeError("Server group did not drain; stopped and refusing next model")
                    groups.discard(server_pid)
                else:
                    server_stopped = final_state.get("server_stopped") is True
                entry["driver_stopped"] = stopped
                entry["server_stopped"] = server_stopped
                entry["elapsed_seconds"] = time.monotonic() - phase_started
                checkpoint()
                if not stopped or not server_stopped:
                    raise RuntimeError("019 owned processes did not stop cleanly")
            entry["status"] = "complete"
            checkpoint()
        report["comparison"] = compare_phase_reports([output / p / "phase-report.json" for p in config["phase_order"]])
        save(output / "comparison.json", report["comparison"])
        report["status"] = "complete"
        return report
    except BaseException as error:
        report["status"] = "failed"
        report["error"] = {"type": type(error).__name__, "message": str(error)}
        raise
    finally:
        cleanup_errors = []
        for driver in drivers:
            if driver.poll() is None:
                try:
                    if not stop_group(driver, grace=20):
                        cleanup_errors.append({"driver_pid": driver.pid, "error": "driver_not_stopped"})
                except Exception as error:
                    cleanup_errors.append({"driver_pid": driver.pid, "error": repr(error)})
        # A failed state-file read or cleanup exception must not bypass the
        # separate server PGID. Only IDs registered by this session are used.
        for pgid in list(groups):
            try:
                os.killpg(pgid, signal.SIGTERM)
                until = time.monotonic() + 3
                while time.monotonic() < until:
                    time.sleep(.1)
                    os.killpg(pgid, 0)
                os.killpg(pgid, signal.SIGKILL)
                time.sleep(.5)
                os.killpg(pgid, 0)
                cleanup_errors.append({"pgid": pgid, "error": "group_still_observed_after_kill"})
            except ProcessLookupError:
                groups.discard(pgid)
            except Exception as error:
                cleanup_errors.append({"pgid": pgid, "error": repr(error)})
        report["cleanup_errors"] = cleanup_errors
        if cleanup_errors:
            report["status"] = "failed_cleanup"
        if monitor is not None:
            report["gpu_sampling"] = monitor.stop()
        done.set()
        sampler.join(timeout=2)
        checkpoint()


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", type=Path, required=True)
    parser.add_argument("--payload-root", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--input-root", default="/kaggle/input")
    args = parser.parse_args()
    report = run(args.config, args.payload_root, args.output, args.input_root)
    print(json.dumps(report, indent=2))
    if report["status"] not in {"complete", "cpu_preflight_complete"}:
        raise SystemExit(1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/019-code/payload-manifest.json
{
  "sha256": {
    "config.json": "20442e3b2a445a65729d308b5f758b67ce499a2d6142840bc207cbffc9ce4652",
    "weight-contract.json": "df08476ffd8b0c5f4bd85491b6dc0c6db34bc67ccf7652f0a0d751bd6a3d810b",
    "suite.json": "a2cbdcc02cb48ae73c64016cc45b2af0c0e54315efeedac328d1b9d473f8792c",
    "launch_server.py": "173d276739cda7cd65f53b4c13afac4e1ec51ee1f08ff5f086984c830b85d3bf",
    "src/arc3_research/__init__.py": "269d658595dd81051d2a0acc4101d500c41f2598a480c32433a26f449b9830bb",
    "src/arc3_research/experiment_config.py": "df2db63b55ce1f2ae261758f8725927f51050a0a867ff0c30d9a4094b152ca4a",
    "src/arc3_research/session_runner.py": "e2bf61508b17d57a8eb4dc5e557908854fe6091d511d6440674120ef2e1da3bb",
    "src/arc3_research/ple_weights.py": "1034c18c4105b2ee497ac7e5b1ff519a6e46cba4135c5c89612d8943a6e87cb5",
    "src/arc3_research/ple_microbench.py": "5a77861936f76bdfd9cc5fba5efed18ce7b8f2e9f76e837d9b5a89d9e30d15a3",
    "src/arc3_research/ple_session.py": "88aa55d51c54c5cd994dd7db4f5527085d869e1c5d4fe65600786d15f63ecbfa"
  },
  "base_notebook_sha256": "88a379f17c3a4ba8c98715fc23155346ccd6eb289bdafebadfc61a3178f9d9bd",
  "original_precache_sha256": "7ef53c4c2f0a6c2e636be00056ac7ce2a6813c8d6d2ced8e65937e1d6465641a",
  "original_serving_sha256": "c362ce556dc35ed0725c31b3f5c60b86bc83a2d94152fd1a109116813a4b5cf8",
  "changes_to_serving": [
    "same explicit server seed in all phases",
    "installed source identity gate",
    "owned process cleanup and readiness report",
    "join checkpoint prefetch before requests"
  ]
}


## Execute the frozen checks

Failure stops this run; no automatic retry or promotion.

In [ ]:
environment_019 = os.environ.copy()
environment_019['PYTHONPATH'] = str(ROOT_019 / 'src')
subprocess.run([sys.executable, '-m', 'arc3_research.ple_session',
    '--config', str(ROOT_019 / 'config.json'), '--payload-root', str(ROOT_019),
    '--output', '/kaggle/working/019-results'], env=environment_019, check=True)


## Recorded outcome

Read the actual status and evidence; this notebook does not claim an unmeasured benefit.

In [ ]:
result_019 = json.loads(Path('/kaggle/working/019-results/session.json').read_text())
print(json.dumps({k: result_019.get(k) for k in ['status','elapsed_seconds','weight_audit_passed','gpu_started']}, indent=2))
for item_019 in result_019.get('phases', []):
    print(item_019['phase'], item_019['status'], item_019.get('startup_seconds'), item_019.get('server_stopped'))
